In [1]:
import pandas as pd
from pyfaidx import Fasta
import re
import sys

WINDOW = 5000
TARGET_LEN = (WINDOW * 2) + 1
PAD_CHAR = 'N'

# --- 2. Các hàm tiện ích ---
def normalize_chrom(chrom):
    """Chuẩn hóa tên chromosome."""
    chrom = str(chrom).strip()
    if chrom.startswith("NC_"):
        num = chrom.split("_")[-1].split(".")[0]
        if num == "000001": return "1"
        elif num == "000002": return "2"
        elif num == "000023": return "X"
        elif num == "000024": return "Y"
        elif num == "000025": return "MT"
        else:
            try:
                num_int = int(num)
                return str(num_int)
            except:
                return chrom
    return chrom.replace("chr", "")

def normalize_chromosome_output(chrom_name):
    """
    Dùng để chuẩn hóa tên chromosome trước khi xuất file CSV.
    Chuyển về dạng UCSC (chr1, chr2...) và loại bỏ contig rác.
    """
    chrom_str = str(chrom_name).strip()

    # Lọc bỏ các contig không xác định (Unplaced/random/alt)
    if 'Un' in chrom_str or 'random' in chrom_str or 'alt' in chrom_str:
        return None # Sẽ trả về None để ta drop hàng này sau

    # Xử lý các trường hợp đặc biệt (MT -> chrM)
    if chrom_str.upper() == 'MT':
        return 'chrM'

    # Nếu nó đã có 'chr' rồi thì thôi
    if chrom_str.startswith('chr'):
        return chrom_str

    # Nếu nó là số (1, 2, X, Y) thì thêm 'chr'
    return 'chr' + chrom_str

def parse_hgvsc_offset(hgvsc_string):
    """Hướng 1: Regex chặt chẽ hơn, yêu cầu dấu +/- phải đứng sau 1 con số"""
    if not isinstance(hgvsc_string, str):
        return None
    
    # Regex giải thích:
    # c\.       : Bắt đầu bằng c.
    # .*?       : Các ký tự ở giữa
    # (?<=\d|\*) : LOOKBEHIND - Ký tự đứng ngay trước dấu +/- PHẢI là số hoặc dấu * (cho 3'UTR)
    # ([+-])    : Nhóm 1 (Dấu)
    # (\d+)     : Nhóm 2 (Giá trị Offset)
    match = re.search(r'c\..*?(?<=\d|\*)([+-])(\d+)', hgvsc_string)
    
    if match:
        sign = match.group(1)
        value = int(match.group(2))
        return -value if sign == '-' else value
    return None

def get_ref_seq(genome, chrom, center_1based, window):
    """Lấy chuỗi DNA 601bp, tự động pad 'N'."""
    target_len = (window * 2) + 1
    start_0based = center_1based - 1 - window
    end_0based = center_1based + window 
    
    try:
        seq = genome[chrom][start_0based:end_0based].upper()
        pad_left = max(0, -start_0based)
        pad_right = max(0, end_0based - len(genome[chrom]))
        final_seq = (PAD_CHAR * pad_left) + seq + (PAD_CHAR * pad_right)
        
        if len(final_seq) != target_len:
             print(f"Lỗi độ dài khi get_ref_seq: {chrom}:{center_1based}. Dự kiến {target_len}, nhận {len(final_seq)}")
             return PAD_CHAR * target_len
        return final_seq
        
    except Exception as e:
        print(f"Lỗi get_ref_seq: {e} tại {chrom}:{center_1based}")
        return PAD_CHAR * target_len

def verify_ref_seq_center(genome, chrom, canonical_center, ref_seq, window):
    """(Hàm kiểm tra) Xác minh base ở tâm `ref_seq` khớp với FASTA."""
    try:
        truth_base = genome[chrom][canonical_center - 1].upper()
        test_base = ref_seq[window].upper()
        if truth_base == test_base:
            return True
        else:
            print(f"--- ⚠️ LỖI CĂN GIỮA! ---")
            print(f"  Tọa độ chuẩn: {chrom}:{canonical_center}")
            print(f"  Base 'Sự thật' từ FASTA: {truth_base}")
            print(f"  Base 'Kiểm tra' tại tâm ref_seq[300]: {test_base}")
            return False
    except Exception as e:
        print(f"LỖI KIỂM TRA (Exception): {e} tại {chrom}:{canonical_center}")
        return False

def normalize_centered_sequence(seq, center_index, target_len, pad_char='N' ):
    """Logic "Cắt/Đệm Đối Xứng" (Symmetric Crop/Pad)."""
    window = target_len // 2
    start = center_index - window
    end = center_index + window + 1
    pad_left = max(0, -start)
    pad_right = max(0, end - len(seq))
    crop_left = max(0, start)
    crop_right = min(len(seq), end)
    
    final_seq = (pad_char * pad_left) + seq[crop_left:crop_right] + (pad_char * pad_right)
    
    if len(final_seq) > target_len:
        over = len(final_seq) - target_len
        final_seq = final_seq[over//2 : -(over - over//2)]
    if len(final_seq) < target_len:
        needed = target_len - len(final_seq)
        final_seq += pad_char * needed
            
    return final_seq

In [2]:
# --- 1. Cấu hình ---
fasta_path = r"D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa"
annotated_variant_file = r"D:\variant_data\train1_final.parquet"
output_file = r"D:\variant_data\train1_final.parquet"

# --- 3. Tải Dữ liệu ---
print(f"🧬 Đang tải FASTA: {fasta_path}")
try:
    genome = Fasta(fasta_path, as_raw=True, sequence_always_upper=True)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể tải file FASTA. Lỗi: {e}")
    sys.exit(1)

print(f"📊 Đang đọc file VEP CSV: {annotated_variant_file}")
try:
    df = pd.read_parquet(annotated_variant_file)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể đọc file Parquet. Lỗi: {e}")
    sys.exit(1)

# Khởi tạo các cột mới
df['canonical_center'] = None
df[f'ref_seq_{TARGET_LEN}'] = None
df[f'alt_seq_{TARGET_LEN}'] = None

count_success = 0
count_fail = 0

# --- 4. Vòng lặp Xử lý ---
total_rows = len(df)
print(f"⚙️ Bắt đầu xử lý {total_rows} biến thể...")
for index, row in df.iterrows():
    if (index + 1) % 1000 == 0:
        print(f"  ...Đã xử lý {index + 1} / {total_rows}...")

    try:
        # 1. Đọc dữ liệu (nếu đã qua filter)
        chrom = normalize_chrom(row['CHROM'])
        pos = int(row['POS'])
        ref_vcf = str(row['REF']).upper()
        alt_vcf = str(row['ALT']).upper()
        hgvsc_str = str(row['HGVSc'])
        consequence_str = str(row['Consequence'])
        # Lấy nhãn đầu tiên (nghiêm trọng nhất)
        first_consequence = consequence_str.split(',')[0]
        
        # 2. Tìm Tâm Chuẩn (Canonical Center)
        offset = parse_hgvsc_offset(hgvsc_str)
        canonical_center = 0

        if first_consequence == 'splice_donor_variant' and offset is not None:
            canonical_center = pos - (offset - 1)
        elif first_consequence == 'splice_acceptor_variant' and offset is not None:
            canonical_center = pos - (offset + 1)
        else:
            # Missense, UTR, hoặc Splicing bị lỗi offset -> Dùng POS làm tâm
            canonical_center = pos
        
        # 3. Tạo ref_seq
        ref_seq = get_ref_seq(genome, chrom, canonical_center, WINDOW)
        
        if not verify_ref_seq_center(genome, chrom, canonical_center, ref_seq, WINDOW):
            print(f"  CẢNH BÁO: Bỏ qua {chrom}:{pos} do lỗi xác minh căn giữa.")
            count_fail += 1
            continue
            
        # 4. Tạo alt_seq
        # Tính vị trí tương đối của biến thể so với tâm mới
        relative_pos = WINDOW + (pos - canonical_center)
        # Ghép chuỗi ALT
        alt_seq_dynamic = ref_seq[:relative_pos] + alt_vcf + ref_seq[relative_pos + len(ref_vcf):]
        indel_len_change = len(alt_vcf) - len(ref_vcf)
        alt_center_index = WINDOW
        # Nếu biến thể chính là tâm (Missense) -> relative_pos = WINDOW -> alt_center = WINDOW + change
        if relative_pos <= WINDOW:
            alt_center_index = WINDOW + indel_len_change
            
        final_alt_seq = normalize_centered_sequence(
            alt_seq_dynamic, 
            alt_center_index, 
            TARGET_LEN, 
            PAD_CHAR
        )

        # 5. Cập nhật vào DataFrame
        df.at[index, 'canonical_center'] = canonical_center
        df.at[index, f'ref_seq_{TARGET_LEN}'] = ref_seq
        df.at[index, f'alt_seq_{TARGET_LEN}'] = final_alt_seq
        
        count_success += 1
        
    except Exception as e:
        print(f"Lỗi xử lý hàng {index}: {e}. Dữ liệu hàng: {row.to_dict()}")

# --- 5. Hậu xử lý và Lưu file ---
print("\n🧹 Đang làm sạch dữ liệu cuối cùng...")

# Check data missing
print('Số missing value sau khi map: \n')
df.isnull().sum()

# 5.1. Loại bỏ những hàng không tạo được sequence (giá trị vẫn là None)
df_final = df.dropna(subset=[f'ref_seq_{TARGET_LEN}'])
# 5.2. Chuẩn hóa tên Chromosome output (hàm của bạn)
df_final['CHROM'] = df_final['CHROM'].apply(normalize_chromosome_output)
# 5.3. Loại bỏ các hàng CHROM bị None (do Un/random/alt)
before_clean = len(df_final)
df_final = df_final.dropna(subset=['CHROM'])
after_clean = len(df_final)

print(f" - Tổng input: {total_rows}")
print(f" - Thành công: {count_success}")
print(f" - Thất bại/Lỗi Verify: {count_fail}")
print(f" - Loại bỏ do nhiễm sắc thể rác: {before_clean - after_clean}")
print(f"✅ Tổng số hàng hợp lệ cuối cùng: {len(df_final)}")

# Lưu Kết quả
if not df_final.empty:
    print(f"💾 Đang lưu kết quả vào: {output_file}")
    df_final.to_parquet(output_file, index=False)
    print("🎉 Hoàn thành!")
else:
    print("⚠️ Cảnh báo: File kết quả rỗng!")

df_final 

🧬 Đang tải FASTA: D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa
📊 Đang đọc file VEP CSV: D:\variant_data\train1_final.parquet
⚙️ Bắt đầu xử lý 104385 biến thể...
  ...Đã xử lý 1000 / 104385...
  ...Đã xử lý 2000 / 104385...
  ...Đã xử lý 3000 / 104385...
  ...Đã xử lý 4000 / 104385...
  ...Đã xử lý 5000 / 104385...
  ...Đã xử lý 6000 / 104385...
  ...Đã xử lý 7000 / 104385...
  ...Đã xử lý 8000 / 104385...
  ...Đã xử lý 9000 / 104385...
  ...Đã xử lý 10000 / 104385...
  ...Đã xử lý 11000 / 104385...
  ...Đã xử lý 12000 / 104385...
  ...Đã xử lý 13000 / 104385...
  ...Đã xử lý 14000 / 104385...
  ...Đã xử lý 15000 / 104385...
  ...Đã xử lý 16000 / 104385...
  ...Đã xử lý 17000 / 104385...
  ...Đã xử lý 18000 / 104385...
  ...Đã xử lý 19000 / 104385...
  ...Đã xử lý 20000 / 104385...
  ...Đã xử lý 21000 / 104385...
  ...Đã xử lý 22000 / 104385...
  ...Đã xử lý 23000 / 104385...
  ...Đã xử lý 24000 / 104385...
  ...Đã xử lý 25000 / 104385...
  ...Đã xử lý 26000 / 104385...


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,ref_seq_301,alt_seq_301,ref_seq_601,alt_seq_601,ref_seq_1001,alt_seq_1001,ref_seq_2001,alt_seq_2001,ref_seq_10001,alt_seq_10001
0,1_1014042_G_A,446939.0,single nucleotide variant,NM_005101.4(ISG15):c.62G>A (p.Ser21Asn),9636.0,ISG15,HGNC:4053,Benign/Likely benign,0.0,"MONDO:MONDO:0014502,MedGen:C4015293,OMIM:61612...",...,CTGGGTACAGGAGCTCGCCCTGCAGCCAGTGCCTTGTGTGTGGTGG...,CTGGGTACAGGAGCTCGCCCTGCAGCCAGTGCCTTGTGTGTGGTGG...,GGTGGGGCACAGAGGGGCACCCTAGCAGGTAAAGGGAGGCCACGGG...,GGTGGGGCACAGAGGGGCACCCTAGCAGGTAAAGGGAGGCCACGGG...,GTGCCGTGGCCCACAGCCCACAGCCCACAGCCATGGTAAGGCAGAT...,GTGCCGTGGCCCACAGCCCACAGCCCACAGCCATGGTAAGGCAGAT...,AACAACAACAAAAAAACACCAATCTGAGCAAATACTGCCCTAAACC...,AACAACAACAAAAAAACACCAATCTGAGCAAATACTGCCCTAAACC...,AAACTCCCAGGCTCAAGCGATCCTCTAGGCTTTGCTTCCAAAGTGC...,AAACTCCCAGGCTCAAGCGATCCTCTAGGCTTTGCTTCCAAAGTGC...
1,1_1014228_G_A,389314.0,single nucleotide variant,NM_005101.4(ISG15):c.248G>A (p.Ser83Asn),9636.0,ISG15,HGNC:4053,Benign,0.0,"MedGen:CN169374|MONDO:MONDO:0014502,MedGen:C40...",...,CCCAGAAGATCGGCGTGCACGCCTTCCAGCAGCGTCTGGCTGTCCA...,CCCAGAAGATCGGCGTGCACGCCTTCCAGCAGCGTCTGGCTGTCCA...,TGTGTGGTGGGCCTGGGGCTGGCGCCGCAGTCTCTGAACCTGTGTG...,TGTGTGGTGGGCCTGGGGCTGGCGCCGCAGTCTCTGAACCTGTGTG...,CCTGAGATCCTCAGGGTGGGGCACAGAGGGGCACCCTAGCAGGTAA...,CCTGAGATCCTCAGGGTGGGGCACAGAGGGGCACCCTAGCAGGTAA...,GTGAAGGATCTGGAATGCGCGATATTTAGGTGTTTCCAGGGTGTTG...,GTGAAGGATCTGGAATGCGCGATATTTAGGTGTTTCCAGGGTGTTG...,GCAACCCCAGGAAACTAAAGCAAGTGACTGCAGGTCTGGTCACCTT...,GCAACCCCAGGAAACTAAAGCAAGTGACTGCAGGTCTGGTCACCTT...
2,1_1014401_G_A,NaN,None,None,NaN,None,None,None,NaN,None,...,GGCCGCAGCAGCACCTACGAGGTACGGCTGACGCAGACCGTGGCCC...,GGCCGCAGCAGCACCTACGAGGTACGGCTGACGCAGACCGTGGCCC...,TTCCAGCAGCGTCTGGCTGTCCACCCGAGCGGTGTGGCGCTGCAGG...,TTCCAGCAGCGTCTGGCTGTCCACCCGAGCGGTGTGGCGCTGCAGG...,GGAGCTCGCCCTGCAGCCAGTGCCTTGTGTGTGGTGGGCCTGGGGC...,GGAGCTCGCCCTGCAGCCAGTGCCTTGTGTGTGGTGGGCCTGGGGC...,TGAAGCCAAATTTGGCCACCAGCGCAGGCTCGGCGGCACGCCCCCT...,TGAAGCCAAATTTGGCCACCAGCGCAGGCTCGGCGGCACGCCCCCT...,GCCGGGCGCGGTGGCTCACGCCTGTAATCTCAGCACTTTGGGAGGA...,GCCGGGCGCGGTGGCTCACGCCTGTAATCTCAGCACTTTGGGAGGA...
3,1_1014471_G_C,446981.0,single nucleotide variant,NM_005101.4(ISG15):c.491G>C (p.Arg164Pro),9636.0,ISG15,HGNC:4053,Likely benign,0.0,"MONDO:MONDO:0014502,MedGen:C4015293,OMIM:61612...",...,TGGAGGGTGTGCAGGACGACCTGTTCTGGCTGACCTTCGAGGGGAA...,TGGAGGGTGTGCAGGACGACCTGTTCTGGCTGACCTTCGAGGGGAA...,GCCTGGGCCCCGGCAGCACGGTCCTGCTGGTGGTGGACAAATGCGA...,GCCTGGGCCCCGGCAGCACGGTCCTGCTGGTGGTGGACAAATGCGA...,GTGACGCCTGCAGGGCTGGGACCTGACGGTGAAGATGCTGGCGGGC...,GTGACGCCTGCAGGGCTGGGACCTGACGGTGAAGATGCTGGCGGGC...,TAGGGCCGGTGCTGCCTGCCGAAGCCGGCGGCTGAGAGGCAGCGAA...,TAGGGCCGGTGCTGCCTGCCGAAGCCGGCGGCTGAGAGGCAGCGAA...,GGAGATCGAGACCATCCTGGGTAACACGGTGAAACCCCGTCTCTAC...,GGAGATCGAGACCATCCTGGGTAACACGGTGAAACCCCGTCTCTAC...
4,1_1020183_G_C,364282.0,single nucleotide variant,NM_198576.4(AGRN):c.11G>C (p.Arg4Pro),375790.0,AGRN,HGNC:329,Benign,0.0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",...,GAGGGGGAGGGGCGCCGCCCTGGCCCCGCCCCCCCCCGCCCGCCCG...,GAGGGGGAGGGGCGCCGCCCTGGCCCCGCCCCCCCCCGCCCGCCCG...,GGCGCTGCGTCCCGGGGCTTTGTTCGCGGCGGCGCGGGTCGCCGGC...,GGCGCTGCGTCCCGGGGCTTTGTTCGCGGCGGCGCGGGTCGCCGGC...,CCTGCCGCCCGGCTCCGCGCCGCCCGGAGCCCCTCGGCAGTCCCGT...,CCTGCCGCCCGGCTCCGCGCCGCCCGGAGCCCCTCGGCAGTCCCGT...,CACCCGGTGTCAAGAGATCTTGCTGCCTCCGCCTCCCAGAGTGCTG...,CACCCGGTGTCAAGAGATCTTGCTGCCTCCGCCTCCCAGAGTGCTG...,CACCGTCCAGGAAGCAGAGCTTGTCCCAGGTGGGGGGCCCCCAATT...,CACCGTCCAGGAAGCAGAGCTTGTCCCAGGTGGGGGGCCCCCAATT...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104380,X_155511709_C_T,2821775.0,single nucleotide variant,NM_018196.4(TMLHE):c.722G>A (p.Arg241Gln),55217.0,TMLHE,HGNC:18308,Likely benign,0.0,MedGen:C3661900,...,CAAATCTGCAGGAGCAGATATGATCTTGGTTCGAACAGAATGGAAA...,CAAATCTGCAGGAGCAGATATGATCTTGGTTCGAACAGAATGGAAA...,GAGACAATTAAATCTCTCTTTTTTTTTTTTATAAATTACCCAGTCT...,GAGACAATTAAATCTCTCTTTTTTTTTTTTATAAATTACCCAGTCT...,CT

In [3]:
# --- 1. Cấu hình ---
fasta_path = r"D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa"
annotated_variant_file = r"D:\variant_data\train2_final.parquet"
output_file = r"D:\variant_data\train2_final.parquet"

# --- 3. Tải Dữ liệu ---
print(f"🧬 Đang tải FASTA: {fasta_path}")
try:
    genome = Fasta(fasta_path, as_raw=True, sequence_always_upper=True)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể tải file FASTA. Lỗi: {e}")
    sys.exit(1)

print(f"📊 Đang đọc file VEP CSV: {annotated_variant_file}")
try:
    df = pd.read_parquet(annotated_variant_file)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể đọc file Parquet. Lỗi: {e}")
    sys.exit(1)

# Khởi tạo các cột mới
df['canonical_center'] = None
df[f'ref_seq_{TARGET_LEN}'] = None
df[f'alt_seq_{TARGET_LEN}'] = None

count_success = 0
count_fail = 0

# --- 4. Vòng lặp Xử lý ---
total_rows = len(df)
print(f"⚙️ Bắt đầu xử lý {total_rows} biến thể...")
for index, row in df.iterrows():
    if (index + 1) % 1000 == 0:
        print(f"  ...Đã xử lý {index + 1} / {total_rows}...")

    try:
        # 1. Đọc dữ liệu (nếu đã qua filter)
        chrom = normalize_chrom(row['CHROM'])
        pos = int(row['POS'])
        ref_vcf = str(row['REF']).upper()
        alt_vcf = str(row['ALT']).upper()
        hgvsc_str = str(row['HGVSc'])
        consequence_str = str(row['Consequence'])
        # Lấy nhãn đầu tiên (nghiêm trọng nhất)
        first_consequence = consequence_str.split(',')[0]
        
        # 2. Tìm Tâm Chuẩn (Canonical Center)
        offset = parse_hgvsc_offset(hgvsc_str)
        canonical_center = 0

        if first_consequence == 'splice_donor_variant' and offset is not None:
            canonical_center = pos - (offset - 1)
        elif first_consequence == 'splice_acceptor_variant' and offset is not None:
            canonical_center = pos - (offset + 1)
        else:
            # Missense, UTR, hoặc Splicing bị lỗi offset -> Dùng POS làm tâm
            canonical_center = pos
        
        # 3. Tạo ref_seq
        ref_seq = get_ref_seq(genome, chrom, canonical_center, WINDOW)
        
        if not verify_ref_seq_center(genome, chrom, canonical_center, ref_seq, WINDOW):
            print(f"  CẢNH BÁO: Bỏ qua {chrom}:{pos} do lỗi xác minh căn giữa.")
            count_fail += 1
            continue
            
        # 4. Tạo alt_seq
        # Tính vị trí tương đối của biến thể so với tâm mới
        relative_pos = WINDOW + (pos - canonical_center)
        # Ghép chuỗi ALT
        alt_seq_dynamic = ref_seq[:relative_pos] + alt_vcf + ref_seq[relative_pos + len(ref_vcf):]
        indel_len_change = len(alt_vcf) - len(ref_vcf)
        alt_center_index = WINDOW
        # Nếu biến thể chính là tâm (Missense) -> relative_pos = WINDOW -> alt_center = WINDOW + change
        if relative_pos <= WINDOW:
            alt_center_index = WINDOW + indel_len_change
            
        final_alt_seq = normalize_centered_sequence(
            alt_seq_dynamic, 
            alt_center_index, 
            TARGET_LEN, 
            PAD_CHAR
        )

        # 5. Cập nhật vào DataFrame
        df.at[index, 'canonical_center'] = canonical_center
        df.at[index, f'ref_seq_{TARGET_LEN}'] = ref_seq
        df.at[index, f'alt_seq_{TARGET_LEN}'] = final_alt_seq
        
        count_success += 1
        
    except Exception as e:
        print(f"Lỗi xử lý hàng {index}: {e}. Dữ liệu hàng: {row.to_dict()}")

# --- 5. Hậu xử lý và Lưu file ---
print("\n🧹 Đang làm sạch dữ liệu cuối cùng...")

# Check data missing
print('Số missing value sau khi map: \n')
df.isnull().sum()

# 5.1. Loại bỏ những hàng không tạo được sequence (giá trị vẫn là None)
df_final = df.dropna(subset=[f'ref_seq_{TARGET_LEN}'])
# 5.2. Chuẩn hóa tên Chromosome output (hàm của bạn)
df_final['CHROM'] = df_final['CHROM'].apply(normalize_chromosome_output)
# 5.3. Loại bỏ các hàng CHROM bị None (do Un/random/alt)
before_clean = len(df_final)
df_final = df_final.dropna(subset=['CHROM'])
after_clean = len(df_final)

print(f" - Tổng input: {total_rows}")
print(f" - Thành công: {count_success}")
print(f" - Thất bại/Lỗi Verify: {count_fail}")
print(f" - Loại bỏ do nhiễm sắc thể rác: {before_clean - after_clean}")
print(f"✅ Tổng số hàng hợp lệ cuối cùng: {len(df_final)}")

# Lưu Kết quả
if not df_final.empty:
    print(f"💾 Đang lưu kết quả vào: {output_file}")
    df_final.to_parquet(output_file, index=False)
    print("🎉 Hoàn thành!")
else:
    print("⚠️ Cảnh báo: File kết quả rỗng!")

df_final 

🧬 Đang tải FASTA: D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa
📊 Đang đọc file VEP CSV: D:\variant_data\train2_final.parquet
⚙️ Bắt đầu xử lý 67990 biến thể...
  ...Đã xử lý 1000 / 67990...
  ...Đã xử lý 2000 / 67990...
  ...Đã xử lý 3000 / 67990...
  ...Đã xử lý 4000 / 67990...
  ...Đã xử lý 5000 / 67990...
  ...Đã xử lý 6000 / 67990...
  ...Đã xử lý 7000 / 67990...
  ...Đã xử lý 8000 / 67990...
  ...Đã xử lý 9000 / 67990...
  ...Đã xử lý 10000 / 67990...
  ...Đã xử lý 11000 / 67990...
  ...Đã xử lý 12000 / 67990...
  ...Đã xử lý 13000 / 67990...
  ...Đã xử lý 14000 / 67990...
  ...Đã xử lý 15000 / 67990...
  ...Đã xử lý 16000 / 67990...
  ...Đã xử lý 17000 / 67990...
  ...Đã xử lý 18000 / 67990...
  ...Đã xử lý 19000 / 67990...
  ...Đã xử lý 20000 / 67990...
  ...Đã xử lý 21000 / 67990...
  ...Đã xử lý 22000 / 67990...
  ...Đã xử lý 23000 / 67990...
  ...Đã xử lý 24000 / 67990...
  ...Đã xử lý 25000 / 67990...
  ...Đã xử lý 26000 / 67990...
  ...Đã xử lý 27000 / 67990

,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,ref_seq_301,alt_seq_301,ref_seq_601,alt_seq_601,ref_seq_1001,alt_seq_1001,ref_seq_2001,alt_seq_2001,ref_seq_10001,alt_seq_10001
0,1_1020183_G_C,364282.0,single nucleotide variant,NM_198576.4(AGRN):c.11G>C (p.Arg4Pro),375790.0,AGRN,HGNC:329,Benign,0.0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",...,GAGGGGGAGGGGCGCCGCCCTGGCCCCGCCCCCCCCCGCCCGCCCG...,GAGGGGGAGGGGCGCCGCCCTGGCCCCGCCCCCCCCCGCCCGCCCG...,GGCGCTGCGTCCCGGGGCTTTGTTCGCGGCGGCGCGGGTCGCCGGC...,GGCGCTGCGTCCCGGGGCTTTGTTCGCGGCGGCGCGGGTCGCCGGC...,CCTGCCGCCCGGCTCCGCGCCGCCCGGAGCCCCTCGGCAGTCCCGT...,CCTGCCGCCCGGCTCCGCGCCGCCCGGAGCCCCTCGGCAGTCCCGT...,CACCCGGTGTCAAGAGATCTTGCTGCCTCCGCCTCCCAGAGTGCTG...,CACCCGGTGTCAAGAGATCTTGCTGCCTCCGCCTCCCAGAGTGCTG...,CACCGTCCAGGAAGCAGAGCTTGTCCCAGGTGGGGGGCCCCCAATT...,CACCGTCCAGGAAGCAGAGCTTGTCCCAGGTGGGGGGCCCCCAATT...
1,1_1022383_C_G,NaN,None,None,NaN,None,None,None,NaN,None,...,CCTGGTGGCCCGGGAGAGCCTGCTGGACGGCGGCAACAAGGTGGTG...,CCTGGTGGCCCGGGAGAGCCTGCTGGACGGCGGCAACAAGGTGGTG...,CACATCTCTGCCCAGGGCTTGAGTCTACTGTGGACATTTGCCCTAA...,CACATCTCTGCCCAGGGCTTGAGTCTACTGTGGACATTTGCCCTAA...,CCAAGCCAGGTGGGCCCCCTTCCCAAATTCGGCCTTAGCTGATAGG...,CCAAGCCAGGTGGGCCCCCTTCCCAAATTCGGCCTTAGCTGATAGG...,TCTGCCTGCGCAACTGGAGGGGGTCCCTGCGGATGCCGCAGTCCTG...,TCTGCCTGCGCAACTGGAGGGGGTCCCTGCGGATGCCGCAGTCCTG...,AAGGCAGCGCGTCTGGATTTTTTGTTCCTCCCGGTGGGCTCGTGGT...,AAGGCAGCGCGTCTGGATTTTTTGTTCCTCCCGGTGGGCTCGTGGT...
2,1_1035307_C_T,446942.0,single nucleotide variant,NM_198576.4(AGRN):c.494C>T (p.Pro165Leu),375790.0,AGRN,HGNC:329,Likely benign,0.0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",...,TGGGGCTAGCGGTGGGGGGGGGGGGGTGGGCAGGGGTGCCCCTTTC...,TGGGGCTAGCGGTGGGGGGGGGGGGGTGGGCAGGGGTGCCCCTTTC...,TCCTGCCTGCACCCCTGTGGCTGGGGCCCCATCTGACAGGGGTCAG...,TCCTGCCTGCACCCCTGTGGCTGGGGCCCCATCTGACAGGGGTCAG...,TTCCCGGGGATGGGATGGGTCACTCTGCGGACCCCTCGGTCCCTGG...,TTCCCGGGGATGGGATGGGTCACTCTGCGGACCCCTCGGTCCCTGG...,CTACCTCGGAGCCCACATTTGGGATTTCTACTCCGGGAGAATTCTG...,CTACCTCGGAGCCCACATTTGGGATTTCTACTCCGGGAGAATTCTG...,GTGTGTGCAGTGCATGGTGCTGTGAGATCAGCGTGTGTGTGTGCAG...,GTGTGTGCAGTGCATGGTGCTGTGAGATCAGCGTGTGTGTGTGCAG...
3,1_1041218_C_T,249307.0,single nucleotide variant,NM_198576.4(AGRN):c.773C>T (p.Thr258Ile),375790.0,AGRN,HGNC:329,Benign/Likely benign,0.0,"MedGen:CN169374|MONDO:MONDO:0014052,MedGen:C38...",...,GGGGAGGAGCGGGGCTGGGAGGGGCCTGGGGGGCGGAGCGGGGCGG...,GGGGAGGAGCGGGGCTGGGAGGGGCCTGGGGGGCGGAGCGGGGCGG...,GCGGGGCCTATGAGATGGAGCGAGGCTGGGAGGGGCTTCGGGGCCA...,GCGGGGCCTATGAGATGGAGCGAGGCTGGGAGGGGCTTCGGGGCCA...,GGGCCGGGCCGGGCGTCCTGCGTCTGCAAGAAGAGCCCGTGCCCCA...,GGGCCGGGCCGGGCGTCCTGCGTCTGCAAGAAGAGCCCGTGCCCCA...,CGGTGAGGGGCTCCGGTGCCGATGTGGAAGGAGCTCCTGAGCAGAA...,CGGTGAGGGGCTCCGGTGCCGATGTGGAAGGAGCTCCTGAGCAGAA...,GATCGTCTCGGCCACTTTCCAGGTCTTGGGAGTGGGGGATGATGGC...,GATCGTCTCGGCCACTTTCCAGGTCTTGGGAGTGGGGGATGATGGC...
4,1_1041583_A_G,133740.0,single nucleotide variant,NM_198576.4(AGRN):c.1058A>G (p.Gln353Arg),375790.0,AGRN,HGNC:329,Benign,0.0,MedGen:CN169374|MedGen:C3661900|MONDO:MONDO:00...,...,TGGGCGCGCGGCGACAGCGTCCTGACTCCTGCCCTCGACCCCCAGA...,TGGGCGCGCGGCGACAGCGTCCTGACTCCTGCCCTCGACCCCCAGA...,CCCGAGGGGACCGTCTGCGGCAGCGACGGCGCCGACTACCCCGGCG...,CCCGAGGGGACCGTCTGCGGCAGCGACGGCGCCGACTACCCCGGCG...,TGGGAGGGGCCTGGGGGGCGGAGCGGGGCGGGAGCGGGGCGGGAGC...,TGGGAGGGGCCTGGGGGGCGGAGCGGGGCGGGAGCGGGGCGGGAGC...,CGAGCACGGCAAGGTCTCTCAGGCTTGTGGACGTGGGTACGGGCGT...,CGAGCACGGCAAGGTCTCTCAGGCTTGTGGACGTGGGTACGGGCGT...,GCTGGGCCTGGGGTGAGGAGCTGGGCAGCGGGAATGGGGAATGGTG...,GCTGGGCCTGGGGTGAGGAGCTGGGCAGCGGGAATGGGGAATGGTG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67985,X_155506983_C_T,4308127.0,single nucleotide variant,NM_018196.4(TMLHE):c.910G>A (p.Asp304Asn),55217.0,TMLHE,HGNC:18308,Likely benign,0.0,MedGen:CN169374,...,TGTTACATTAGAAGGCTAATCTTTGGCAAATATATTACAGTTGGGG...,TGTTACATTAGAAGGCTAATCTTTGGCAAATATATTACAGTTGGGG...,ACCCCACCCCTAAAATAAACCTAGTATTCTTTGGTTGGTGGCTATT...,ACCCCACCCCTAAAATAAACCTAGTATTCTTTGGTTGGTGGCTATT...,GATA

In [4]:
# --- 1. Cấu hình ---
fasta_path = r"D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa"
annotated_variant_file = r"D:\variant_data\train3_final.parquet"
output_file = r"D:\variant_data\train3_final.parquet"

# --- 3. Tải Dữ liệu ---
print(f"🧬 Đang tải FASTA: {fasta_path}")
try:
    genome = Fasta(fasta_path, as_raw=True, sequence_always_upper=True)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể tải file FASTA. Lỗi: {e}")
    sys.exit(1)

print(f"📊 Đang đọc file VEP CSV: {annotated_variant_file}")
try:
    df = pd.read_parquet(annotated_variant_file)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể đọc file Parquet. Lỗi: {e}")
    sys.exit(1)

# Khởi tạo các cột mới
df['canonical_center'] = None
df[f'ref_seq_{TARGET_LEN}'] = None
df[f'alt_seq_{TARGET_LEN}'] = None

count_success = 0
count_fail = 0

# --- 4. Vòng lặp Xử lý ---
total_rows = len(df)
print(f"⚙️ Bắt đầu xử lý {total_rows} biến thể...")
for index, row in df.iterrows():
    if (index + 1) % 1000 == 0:
        print(f"  ...Đã xử lý {index + 1} / {total_rows}...")

    try:
        # 1. Đọc dữ liệu (nếu đã qua filter)
        chrom = normalize_chrom(row['CHROM'])
        pos = int(row['POS'])
        ref_vcf = str(row['REF']).upper()
        alt_vcf = str(row['ALT']).upper()
        hgvsc_str = str(row['HGVSc'])
        consequence_str = str(row['Consequence'])
        # Lấy nhãn đầu tiên (nghiêm trọng nhất)
        first_consequence = consequence_str.split(',')[0]
        
        # 2. Tìm Tâm Chuẩn (Canonical Center)
        offset = parse_hgvsc_offset(hgvsc_str)
        canonical_center = 0

        if first_consequence == 'splice_donor_variant' and offset is not None:
            canonical_center = pos - (offset - 1)
        elif first_consequence == 'splice_acceptor_variant' and offset is not None:
            canonical_center = pos - (offset + 1)
        else:
            # Missense, UTR, hoặc Splicing bị lỗi offset -> Dùng POS làm tâm
            canonical_center = pos
        
        # 3. Tạo ref_seq
        ref_seq = get_ref_seq(genome, chrom, canonical_center, WINDOW)
        
        if not verify_ref_seq_center(genome, chrom, canonical_center, ref_seq, WINDOW):
            print(f"  CẢNH BÁO: Bỏ qua {chrom}:{pos} do lỗi xác minh căn giữa.")
            count_fail += 1
            continue
            
        # 4. Tạo alt_seq
        # Tính vị trí tương đối của biến thể so với tâm mới
        relative_pos = WINDOW + (pos - canonical_center)
        # Ghép chuỗi ALT
        alt_seq_dynamic = ref_seq[:relative_pos] + alt_vcf + ref_seq[relative_pos + len(ref_vcf):]
        indel_len_change = len(alt_vcf) - len(ref_vcf)
        alt_center_index = WINDOW
        # Nếu biến thể chính là tâm (Missense) -> relative_pos = WINDOW -> alt_center = WINDOW + change
        if relative_pos <= WINDOW:
            alt_center_index = WINDOW + indel_len_change
            
        final_alt_seq = normalize_centered_sequence(
            alt_seq_dynamic, 
            alt_center_index, 
            TARGET_LEN, 
            PAD_CHAR
        )

        # 5. Cập nhật vào DataFrame
        df.at[index, 'canonical_center'] = canonical_center
        df.at[index, f'ref_seq_{TARGET_LEN}'] = ref_seq
        df.at[index, f'alt_seq_{TARGET_LEN}'] = final_alt_seq
        
        count_success += 1
        
    except Exception as e:
        print(f"Lỗi xử lý hàng {index}: {e}. Dữ liệu hàng: {row.to_dict()}")

# --- 5. Hậu xử lý và Lưu file ---
print("\n🧹 Đang làm sạch dữ liệu cuối cùng...")

# Check data missing
print('Số missing value sau khi map: \n')
df.isnull().sum()

# 5.1. Loại bỏ những hàng không tạo được sequence (giá trị vẫn là None)
df_final = df.dropna(subset=[f'ref_seq_{TARGET_LEN}'])
# 5.2. Chuẩn hóa tên Chromosome output (hàm của bạn)
df_final['CHROM'] = df_final['CHROM'].apply(normalize_chromosome_output)
# 5.3. Loại bỏ các hàng CHROM bị None (do Un/random/alt)
before_clean = len(df_final)
df_final = df_final.dropna(subset=['CHROM'])
after_clean = len(df_final)

print(f" - Tổng input: {total_rows}")
print(f" - Thành công: {count_success}")
print(f" - Thất bại/Lỗi Verify: {count_fail}")
print(f" - Loại bỏ do nhiễm sắc thể rác: {before_clean - after_clean}")
print(f"✅ Tổng số hàng hợp lệ cuối cùng: {len(df_final)}")

# Lưu Kết quả
if not df_final.empty:
    print(f"💾 Đang lưu kết quả vào: {output_file}")
    df_final.to_parquet(output_file, index=False)
    print("🎉 Hoàn thành!")
else:
    print("⚠️ Cảnh báo: File kết quả rỗng!")

df_final 

🧬 Đang tải FASTA: D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa
📊 Đang đọc file VEP CSV: D:\variant_data\train3_final.parquet
⚙️ Bắt đầu xử lý 23208 biến thể...
  ...Đã xử lý 1000 / 23208...
  ...Đã xử lý 2000 / 23208...
  ...Đã xử lý 3000 / 23208...
  ...Đã xử lý 4000 / 23208...
  ...Đã xử lý 5000 / 23208...
  ...Đã xử lý 6000 / 23208...
  ...Đã xử lý 7000 / 23208...
  ...Đã xử lý 8000 / 23208...
  ...Đã xử lý 9000 / 23208...
  ...Đã xử lý 10000 / 23208...
  ...Đã xử lý 11000 / 23208...
  ...Đã xử lý 12000 / 23208...
  ...Đã xử lý 13000 / 23208...
  ...Đã xử lý 14000 / 23208...
  ...Đã xử lý 15000 / 23208...
  ...Đã xử lý 16000 / 23208...
  ...Đã xử lý 17000 / 23208...
  ...Đã xử lý 18000 / 23208...
  ...Đã xử lý 19000 / 23208...
  ...Đã xử lý 20000 / 23208...
  ...Đã xử lý 21000 / 23208...
  ...Đã xử lý 22000 / 23208...
  ...Đã xử lý 23000 / 23208...

🧹 Đang làm sạch dữ liệu cuối cùng...
Số missing value sau khi map: 

 - Tổng input: 23208
 - Thành công: 23208
 - Thất 

,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,ref_seq_301,alt_seq_301,ref_seq_601,alt_seq_601,ref_seq_1001,alt_seq_1001,ref_seq_2001,alt_seq_2001,ref_seq_10001,alt_seq_10001
0,12_53315378_C_T,2048536.0,single nucleotide variant,NM_015665.6(AAAS):c.356G>A (p.Arg119Gln),8086.0,AAAS,HGNC:13666,Likely benign,0.0,MedGen:C3661900,...,CCCTCTGAAGTCATCACACCCAACCCTGACCTGCAAGGATAGGAAT...,CCCTCTGAAGTCATCACACCCAACCCTGACCTGCAAGGATAGGAAT...,GGGCTTGTGCACTCACCAATTTGTGACTTGGGCAAATTCAGCGATC...,GGGCTTGTGCACTCACCAATTTGTGACTTGGGCAAATTCAGCGATC...,CTGCACCTATCCCTACCCTAGCCCAGAAGCTCACATTCATTTCCAG...,CTGCACCTATCCCTACCCTAGCCCAGAAGCTCACATTCATTTCCAG...,GGGCTTCCAGGCCAGAGACGCCACATTTCGCTGCAGCCGGTGCTTC...,GGGCTTCCAGGCCAGAGACGCCACATTTCGCTGCAGCCGGTGCTTC...,TGAAACCCCGTCTCTACTAAAAGTACAAAAAAATTAGCTGGGCGTG...,TGAAACCCCGTCTCTACTAAAAGTACAAAAAAATTAGCTGGGCGTG...
1,12_53321403_G_C,325881.0,single nucleotide variant,NM_015665.6(AAAS):c.63C>G (p.His21Gln),8086.0,AAAS,HGNC:13666,Likely benign,0.0,MedGen:C3661900|,...,ACCCTGCCCCTGTCACACTGCCTCCTTTCCCCAGTAGTCCCCGACT...,ACCCTGCCCCTGTCACACTGCCTCCTTTCCCCAGTAGTCCCCGACT...,ATTCTCCCTCATTCTCTCCATTCCCTCTAGGCCTCCTCCACAGCTC...,ATTCTCCCTCATTCTCTCCATTCCCTCTAGGCCTCCTCCACAGCTC...,TTATGAAACCATATGACATTTTTAGAAAACACCTGACTCAGCCTCT...,TTATGAAACCATATGACATTTTTAGAAAACACCTGACTCAGCCTCT...,TCTATAATAAGGACTAAAAATTTCATATGATTTAGGATATACAGCT...,TCTATAATAAGGACTAAAAATTTCATATGATTTAGGATATACAGCT...,CGAGATTGTGCCACTGCACTCCAGCCTGGGCGACAGAGTGAGACTC...,CGAGATTGTGCCACTGCACTCCAGCCTGGGCGACAGAGTGAGACTC...
2,12_53314832_C_T,859992.0,single nucleotide variant,NM_015665.6(AAAS):c.464G>A (p.Arg155His),8086.0,AAAS,HGNC:13666,Pathogenic/Likely pathogenic,1.0,"MedGen:C3661900|MONDO:MONDO:0009279,MedGen:C02...",...,CCGGAACCAAGGCTGGCAAGGGAAGGTGATATTGACAAGTCAGAGC...,CCGGAACCAAGGCTGGCAAGGGAAGGTGATATTGACAAGTCAGAGC...,GAAGGATGATCCATCCAGGGGCCAGGGGCACCATAGGACAGGAGGG...,GAAGGATGATCCATCCAGGGGCCAGGGGCACCATAGGACAGGAGGG...,TAAGAATGCAGCTCTGGCAGGCCACAGCCAAGACAGAGGCACTAAG...,TAAGAATGCAGCTCTGGCAGGCCACAGCCAAGACAGAGGCACTAAG...,CTCCTGACCTCAGGTGATCCAACCGCCTCCACATCCCAAAGTGTTG...,CTCCTGACCTCAGGTGATCCAACCGCCTCCACATCCCAAAGTGTTG...,CTCAAATCCAGGTTGTGAAAAAGGAATAAGGATTGTGAAAGTTAAA...,CTCAAATCCAGGTTGTGAAAAAGGAATAAGGATTGTGAAAGTTAAA...
3,12_53321423_G_T,20083.0,single nucleotide variant,NM_015665.6(AAAS):c.43C>A (p.Gln15Lys),8086.0,AAAS,HGNC:13666,Pathogenic,1.0,"MONDO:MONDO:0009279,MedGen:C0271742,OMIM:23155...",...,CCTCCTTTCCCCAGTAGTCCCCGACTCCGCCCCCATGCCTGTAGGT...,CCTCCTTTCCCCAGTAGTCCCCGACTCCGCCCCCATGCCTGTAGGT...,TTCCCTCTAGGCCTCCTCCACAGCTCCTTAACCGTTCCCCCGTTAT...,TTCCCTCTAGGCCTCCTCCACAGCTCCTTAACCGTTCCCCCGTTAT...,TTTAGAAAACACCTGACTCAGCCTCTTTCTTACATAGCAAACGCTC...,TTTAGAAAACACCTGACTCAGCCTCTTTCTTACATAGCAAACGCTC...,TTTCATATGATTTAGGATATACAGCTCTCGTGGAGACAGCCTGAAT...,TTTCATATGATTTAGGATATACAGCTCTCGTGGAGACAGCCTGAAT...,CCAGCCTGGGCGACAGAGTGAGACTCTGCCTCAAAAAAGAAGAAAA...,CCAGCCTGGGCGACAGAGTGAGACTCTGCCTCAAAAAAGAAGAAAA...
4,16_70258214_C_T,466634.0,single nucleotide variant,NM_001605.3(AARS1):c.1996G>A (p.Val666Ile),16.0,AARS1,HGNC:20,Likely benign,0.0,"MONDO:MONDO:0018993,MedGen:C0270914,Orphanet:6...",...,AGCCAGCAGGCCCAGAGGGGTCATCCAGCAACTCGGACACCGGGAC...,AGCCAGCAGGCCCAGAGGGGTCATCCAGCAACTCGGACACCGGGAC...,GGCATTTCTGAGGACTTCACTTCAACCTTAGACAGACAAGAGTTGG...,GGCATTTCTGAGGACTTCACTTCAACCTTAGACAGACAAGAGTTGG...,TTGCTGAATCCCAAATTCACACAAGCCTTCTTGCTAGTTTCAGCCC...,TTGCTGAATCCCAAATTCACACAAGCCTTCTTGCTAGTTTCAGCCC...,ACTCCAGCCTGGGCAATGAAACTCCGTCTCAAAAAAAAAAAAAGAA...,ACTCCAGCCTGGGCAATGAAACTCCGTCTCAAAAAAAAAAAAAGAA...,TCAGCCAACAACCTTAAGACCTAACACTTCCCACTGGTGCGAGGTG...,TCAGCCAACAACCTTAAGACCTAACACTTCCCACTGGTGCGAGGTG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23203,7_76425104_A_G,3409690.0,single nucleotide variant,NM_001110354.2(ZP3):c.140A>G (p.Gln47Arg),7784.0,ZP3,HGNC:13189,Likely benign,0.0,MedGen:C3661900|MedGen:CN169374,...,CTGCAGGTACCATGGAGCTGAGCTATAGGCTCTTCATCTGCCTCCT...,CTGCAGGTACCATGGAGCTGAGCTATAGGCTCTTCATCTGCCTCCT...,AGGTGTTAC

In [5]:
# --- 1. Cấu hình ---
fasta_path = r"D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa"
annotated_variant_file = r"D:\variant_data\val_final.parquet"
output_file = r"D:\variant_data\val_final.parquet"

# --- 3. Tải Dữ liệu ---
print(f"🧬 Đang tải FASTA: {fasta_path}")
try:
    genome = Fasta(fasta_path, as_raw=True, sequence_always_upper=True)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể tải file FASTA. Lỗi: {e}")
    sys.exit(1)

print(f"📊 Đang đọc file VEP CSV: {annotated_variant_file}")
try:
    df = pd.read_parquet(annotated_variant_file)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể đọc file Parquet. Lỗi: {e}")
    sys.exit(1)

# Khởi tạo các cột mới
df['canonical_center'] = None
df[f'ref_seq_{TARGET_LEN}'] = None
df[f'alt_seq_{TARGET_LEN}'] = None

count_success = 0
count_fail = 0

# --- 4. Vòng lặp Xử lý ---
total_rows = len(df)
print(f"⚙️ Bắt đầu xử lý {total_rows} biến thể...")
for index, row in df.iterrows():
    if (index + 1) % 1000 == 0:
        print(f"  ...Đã xử lý {index + 1} / {total_rows}...")

    try:
        # 1. Đọc dữ liệu (nếu đã qua filter)
        chrom = normalize_chrom(row['CHROM'])
        pos = int(row['POS'])
        ref_vcf = str(row['REF']).upper()
        alt_vcf = str(row['ALT']).upper()
        hgvsc_str = str(row['HGVSc'])
        consequence_str = str(row['Consequence'])
        # Lấy nhãn đầu tiên (nghiêm trọng nhất)
        first_consequence = consequence_str.split(',')[0]
        
        # 2. Tìm Tâm Chuẩn (Canonical Center)
        offset = parse_hgvsc_offset(hgvsc_str)
        canonical_center = 0

        if first_consequence == 'splice_donor_variant' and offset is not None:
            canonical_center = pos - (offset - 1)
        elif first_consequence == 'splice_acceptor_variant' and offset is not None:
            canonical_center = pos - (offset + 1)
        else:
            # Missense, UTR, hoặc Splicing bị lỗi offset -> Dùng POS làm tâm
            canonical_center = pos
        
        # 3. Tạo ref_seq
        ref_seq = get_ref_seq(genome, chrom, canonical_center, WINDOW)
        
        if not verify_ref_seq_center(genome, chrom, canonical_center, ref_seq, WINDOW):
            print(f"  CẢNH BÁO: Bỏ qua {chrom}:{pos} do lỗi xác minh căn giữa.")
            count_fail += 1
            continue
            
        # 4. Tạo alt_seq
        # Tính vị trí tương đối của biến thể so với tâm mới
        relative_pos = WINDOW + (pos - canonical_center)
        # Ghép chuỗi ALT
        alt_seq_dynamic = ref_seq[:relative_pos] + alt_vcf + ref_seq[relative_pos + len(ref_vcf):]
        indel_len_change = len(alt_vcf) - len(ref_vcf)
        alt_center_index = WINDOW
        # Nếu biến thể chính là tâm (Missense) -> relative_pos = WINDOW -> alt_center = WINDOW + change
        if relative_pos <= WINDOW:
            alt_center_index = WINDOW + indel_len_change
            
        final_alt_seq = normalize_centered_sequence(
            alt_seq_dynamic, 
            alt_center_index, 
            TARGET_LEN, 
            PAD_CHAR
        )

        # 5. Cập nhật vào DataFrame
        df.at[index, 'canonical_center'] = canonical_center
        df.at[index, f'ref_seq_{TARGET_LEN}'] = ref_seq
        df.at[index, f'alt_seq_{TARGET_LEN}'] = final_alt_seq
        
        count_success += 1
        
    except Exception as e:
        print(f"Lỗi xử lý hàng {index}: {e}. Dữ liệu hàng: {row.to_dict()}")

# --- 5. Hậu xử lý và Lưu file ---
print("\n🧹 Đang làm sạch dữ liệu cuối cùng...")

# Check data missing
print('Số missing value sau khi map: \n')
df.isnull().sum()

# 5.1. Loại bỏ những hàng không tạo được sequence (giá trị vẫn là None)
df_final = df.dropna(subset=[f'ref_seq_{TARGET_LEN}'])
# 5.2. Chuẩn hóa tên Chromosome output (hàm của bạn)
df_final['CHROM'] = df_final['CHROM'].apply(normalize_chromosome_output)
# 5.3. Loại bỏ các hàng CHROM bị None (do Un/random/alt)
before_clean = len(df_final)
df_final = df_final.dropna(subset=['CHROM'])
after_clean = len(df_final)

print(f" - Tổng input: {total_rows}")
print(f" - Thành công: {count_success}")
print(f" - Thất bại/Lỗi Verify: {count_fail}")
print(f" - Loại bỏ do nhiễm sắc thể rác: {before_clean - after_clean}")
print(f"✅ Tổng số hàng hợp lệ cuối cùng: {len(df_final)}")

# Lưu Kết quả
if not df_final.empty:
    print(f"💾 Đang lưu kết quả vào: {output_file}")
    df_final.to_parquet(output_file, index=False)
    print("🎉 Hoàn thành!")
else:
    print("⚠️ Cảnh báo: File kết quả rỗng!")

df_final 

🧬 Đang tải FASTA: D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa
📊 Đang đọc file VEP CSV: D:\variant_data\val_final.parquet
⚙️ Bắt đầu xử lý 13701 biến thể...
  ...Đã xử lý 1000 / 13701...
  ...Đã xử lý 2000 / 13701...
  ...Đã xử lý 3000 / 13701...
  ...Đã xử lý 4000 / 13701...
  ...Đã xử lý 5000 / 13701...
  ...Đã xử lý 6000 / 13701...
  ...Đã xử lý 7000 / 13701...
  ...Đã xử lý 8000 / 13701...
  ...Đã xử lý 9000 / 13701...
  ...Đã xử lý 10000 / 13701...
  ...Đã xử lý 11000 / 13701...
  ...Đã xử lý 12000 / 13701...
  ...Đã xử lý 13000 / 13701...

🧹 Đang làm sạch dữ liệu cuối cùng...
Số missing value sau khi map: 

 - Tổng input: 13701
 - Thành công: 13701
 - Thất bại/Lỗi Verify: 0
 - Loại bỏ do nhiễm sắc thể rác: 0
✅ Tổng số hàng hợp lệ cuối cùng: 13701
💾 Đang lưu kết quả vào: D:\variant_data\val_final.parquet
🎉 Hoàn thành!


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,ref_seq_301,alt_seq_301,ref_seq_601,alt_seq_601,ref_seq_1001,alt_seq_1001,ref_seq_2001,alt_seq_2001,ref_seq_10001,alt_seq_10001
0,4_1022218_C_T,720682.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.95C>T (p.Ala32Val),53834.0,FGFRL1,HGNC:3693,Benign,0.0,MedGen:C3661900|,...,GCTCATCTTGGGCAGGTGACCCAGGTGGAGCTCAGGCCCGAGGTCT...,GCTCATCTTGGGCAGGTGACCCAGGTGGAGCTCAGGCCCGAGGTCT...,CTCGTGTCTCAAAGAGCCAGCCTCTGGGGCCACGGGGCTGCCCCGG...,CTCGTGTCTCAAAGAGCCAGCCTCTGGGGCCACGGGGCTGCCCCGG...,AGGGGCCTGGGGCCTCTTTGGGACCCACCAGGCCCCCCATCCCCCT...,AGGGGCCTGGGGCCTCTTTGGGACCCACCAGGCCCCCCATCCCCCT...,GCAGGGGATGGGGTGGGTTCCTTTCTGCCAGGAAAGGAGGCCTCTG...,GCAGGGGATGGGGTGGGTTCCTTTCTGCCAGGAAAGGAGGCCTCTG...,TGGAAACTTTGGGCCATGCAGGGCCCCCCAGTTCGCAGGCATCCTC...,TGGAAACTTTGGGCCATGCAGGGCCCCCCAGTTCGCAGGCATCCTC...
1,4_1022236_G_A,1949144.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.113G>A (p.Arg38Gln),53834.0,FGFRL1,HGNC:3693,Likely benign,0.0,MedGen:C3661900,...,ACCCAGGTGGAGCTCAGGCCCGAGGTCTGTGCTGGGCCGTGGGTCC...,ACCCAGGTGGAGCTCAGGCCCGAGGTCTGTGCTGGGCCGTGGGTCC...,AGCCTCTGGGGCCACGGGGCTGCCCCGGCCATGAGAGGCTGCTGAC...,AGCCTCTGGGGCCACGGGGCTGCCCCGGCCATGAGAGGCTGCTGAC...,TGGGACCCACCAGGCCCCCCATCCCCCTACTGCCCTTGTCCTTGGG...,TGGGACCCACCAGGCCCCCCATCCCCCTACTGCCCTTGTCCTTGGG...,TCCTTTCTGCCAGGAAAGGAGGCCTCTGCCAGGAAAGGAGGGCTCG...,TCCTTTCTGCCAGGAAAGGAGGCCTCTGCCAGGAAAGGAGGGCTCG...,CAGGGCCCCCCAGTTCGCAGGCATCCTCACTCCACTCTGGCCTCGT...,CAGGGCCCCCCAGTTCGCAGGCATCCTCACTCCACTCTGGCCTCGT...
2,4_1023924_G_A,734353.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.541G>A (p.Asp181Asn),53834.0,FGFRL1,HGNC:3693,Benign/Likely benign,0.0,"MedGen:C3661900|MONDO:MONDO:0008684,MedGen:C19...",...,CTGCATCCCCGTCCTCTGACCTCCACGCCACCCCACCCCGCAGCAC...,CTGCATCCCCGTCCTCTGACCTCCACGCCACCCCACCCCGCAGCAC...,CCTCCGTCTCTCTGCAGATGACATTAGCCCAGGGAAGGAGAGCCTG...,CCTCCGTCTCTCTGCAGATGACATTAGCCCAGGGAAGGAGAGCCTG...,CCGTGCCCATCAGGGTCACCTGCGCCCAGTGTGGGCCAGCGGCCCT...,CCGTGCCCATCAGGGTCACCTGCGCCCAGTGTGGGCCAGCGGCCCT...,ATACCTAGGGACCCGGCACCCCGCCTGCTCCGCTGGGCTCCTGGGT...,ATACCTAGGGACCCGGCACCCCGCCTGCTCCGCTGGGCTCCTGGGT...,GAGGCTGTTTGCTTTAGGATAAGCCTGGCTCCTCTGGCCCAGCTGC...,GAGGCTGTTTGCTTTAGGATAAGCCTGGCTCCTCTGGCCCAGCTGC...
3,4_1024917_C_A,1154604.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.1085C>A (p.Pro362Gln),53834.0,FGFRL1,HGNC:3693,Benign,0.0,MedGen:C3661900,...,CCGTGCTGGCCAGGTCATCTGCCGAGGGAGGGCAGCCCAGGGGCAC...,CCGTGCTGGCCAGGTCATCTGCCGAGGGAGGGCAGCCCAGGGGCAC...,ACACCATGGGCTACAGCTTCCGCAGCGCCTTCCTCACCGTGCTGCC...,ACACCATGGGCTACAGCTTCCGCAGCGCCTTCCTCACCGTGCTGCC...,GCCGGTGATCCAGTGGCTGAAGCGCGTGGAGTACGGCGCCGAGGGC...,GCCGGTGATCCAGTGGCTGAAGCGCGTGGAGTACGGCGCCGAGGGC...,TCGGCCCGACATCACGTGGATGAAGGACGACCAGGCCTTGACGCGC...,TCGGCCCGACATCACGTGGATGAAGGACGACCAGGCCTTGACGCGC...,TCTGGGCACAGCAGGGAGGCGCCGTGTCTCCACTCCCAGCCCCAGC...,TCTGGGCACAGCAGGGAGGCGCCGTGTCTCCACTCCCAGCCCCAGC...
4,4_1024946_G_A,2724602.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.1114G>A (p.Ala372Thr),53834.0,FGFRL1,HGNC:3693,Likely benign,0.0,MedGen:CN169374,...,GGGCAGCCCAGGGGCACCGTCTCCACAGCCCCTGGGATGGGTCTGG...,GGGCAGCCCAGGGGCACCGTCTCCACAGCCCCTGGGATGGGTCTGG...,TTCCTCACCGTGCTGCCAGGTGCGCGGCTGCCACGCCACGCCACAC...,TTCCTCACCGTGCTGCCAGGTGCGCGGCTGCCACGCCACGCCACAC...,AGTACGGCGCCGAGGGCCGCCACAACTCCACCATCGATGTGGGCGG...,AGTACGGCGCCGAGGGCCGCCACAACTCCACCATCGATGTGGGCGG...,ACCAGGCCTTGACGCGCCCAGAGGCCGCTGAGCCCAGGAAGAAGAA...,ACCAGGCCTTGACGCGCCCAGAGGCCGCTGAGCCCAGGAAGAAGAA...,CCACTCCCAGCCCCAGCGGCCAATGATGCCTGTGGGCTCAGAACCT...,CCACTCCCAGCCCCAGCGGCCAATGATGCCTGTGGGCTCAGAACCT...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13696,22_50744150_G_A,3640163.0,single nucleotide variant,NM_001097.3(ACR):c.655G>A (p.Val219Ile),49.0,ACR,HGNC:126,Likely benign,0.0,MedGen:CN169374,...,GGAGGCAGGGCTTTTGTCCGTGTCTCCCGTGATCACTGACCACCGA...,GGAGGCAGGGCTTTTGTCCGTGTCTCCCGTGATCACTGACCACCGA...,TCCTCTGGCCTTGATTTGGATAACATTTCCCCACCTCCTCCCACCA...,TCCTCTGGCCTTGATTTGGATAACATTTCCCCACCTCC

In [6]:
# --- 1. Cấu hình ---
fasta_path = r"D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa"
annotated_variant_file = r"D:\variant_data\test_seq_after_vep_final.parquet"
output_file = r"D:\variant_data\test_seq_after_vep_final.parquet"

# --- 3. Tải Dữ liệu ---
print(f"🧬 Đang tải FASTA: {fasta_path}")
try:
    genome = Fasta(fasta_path, as_raw=True, sequence_always_upper=True)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể tải file FASTA. Lỗi: {e}")
    sys.exit(1)

print(f"📊 Đang đọc file VEP CSV: {annotated_variant_file}")
try:
    df = pd.read_parquet(annotated_variant_file)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể đọc file Parquet. Lỗi: {e}")
    sys.exit(1)

# Khởi tạo các cột mới
df['canonical_center'] = None
df[f'ref_seq_{TARGET_LEN}'] = None
df[f'alt_seq_{TARGET_LEN}'] = None

count_success = 0
count_fail = 0

# --- 4. Vòng lặp Xử lý ---
total_rows = len(df)
print(f"⚙️ Bắt đầu xử lý {total_rows} biến thể...")
for index, row in df.iterrows():
    if (index + 1) % 1000 == 0:
        print(f"  ...Đã xử lý {index + 1} / {total_rows}...")

    try:
        # 1. Đọc dữ liệu (nếu đã qua filter)
        chrom = normalize_chrom(row['CHROM'])
        pos = int(row['POS'])
        ref_vcf = str(row['REF']).upper()
        alt_vcf = str(row['ALT']).upper()
        hgvsc_str = str(row['HGVSc'])
        consequence_str = str(row['Consequence'])
        # Lấy nhãn đầu tiên (nghiêm trọng nhất)
        first_consequence = consequence_str.split(',')[0]
        
        # 2. Tìm Tâm Chuẩn (Canonical Center)
        offset = parse_hgvsc_offset(hgvsc_str)
        canonical_center = 0

        if first_consequence == 'splice_donor_variant' and offset is not None:
            canonical_center = pos - (offset - 1)
        elif first_consequence == 'splice_acceptor_variant' and offset is not None:
            canonical_center = pos - (offset + 1)
        else:
            # Missense, UTR, hoặc Splicing bị lỗi offset -> Dùng POS làm tâm
            canonical_center = pos
        
        # 3. Tạo ref_seq
        ref_seq = get_ref_seq(genome, chrom, canonical_center, WINDOW)
        
        if not verify_ref_seq_center(genome, chrom, canonical_center, ref_seq, WINDOW):
            print(f"  CẢNH BÁO: Bỏ qua {chrom}:{pos} do lỗi xác minh căn giữa.")
            count_fail += 1
            continue
            
        # 4. Tạo alt_seq
        # Tính vị trí tương đối của biến thể so với tâm mới
        relative_pos = WINDOW + (pos - canonical_center)
        # Ghép chuỗi ALT
        alt_seq_dynamic = ref_seq[:relative_pos] + alt_vcf + ref_seq[relative_pos + len(ref_vcf):]
        indel_len_change = len(alt_vcf) - len(ref_vcf)
        alt_center_index = WINDOW
        # Nếu biến thể chính là tâm (Missense) -> relative_pos = WINDOW -> alt_center = WINDOW + change
        if relative_pos <= WINDOW:
            alt_center_index = WINDOW + indel_len_change
            
        final_alt_seq = normalize_centered_sequence(
            alt_seq_dynamic, 
            alt_center_index, 
            TARGET_LEN, 
            PAD_CHAR
        )

        # 5. Cập nhật vào DataFrame
        df.at[index, 'canonical_center'] = canonical_center
        df.at[index, f'ref_seq_{TARGET_LEN}'] = ref_seq
        df.at[index, f'alt_seq_{TARGET_LEN}'] = final_alt_seq
        
        count_success += 1
        
    except Exception as e:
        print(f"Lỗi xử lý hàng {index}: {e}. Dữ liệu hàng: {row.to_dict()}")

# --- 5. Hậu xử lý và Lưu file ---
print("\n🧹 Đang làm sạch dữ liệu cuối cùng...")

# Check data missing
print('Số missing value sau khi map: \n')
df.isnull().sum()

# 5.1. Loại bỏ những hàng không tạo được sequence (giá trị vẫn là None)
df_final = df.dropna(subset=[f'ref_seq_{TARGET_LEN}'])
# 5.2. Chuẩn hóa tên Chromosome output (hàm của bạn)
df_final['CHROM'] = df_final['CHROM'].apply(normalize_chromosome_output)
# 5.3. Loại bỏ các hàng CHROM bị None (do Un/random/alt)
before_clean = len(df_final)
df_final = df_final.dropna(subset=['CHROM'])
after_clean = len(df_final)

print(f" - Tổng input: {total_rows}")
print(f" - Thành công: {count_success}")
print(f" - Thất bại/Lỗi Verify: {count_fail}")
print(f" - Loại bỏ do nhiễm sắc thể rác: {before_clean - after_clean}")
print(f"✅ Tổng số hàng hợp lệ cuối cùng: {len(df_final)}")

# Lưu Kết quả
if not df_final.empty:
    print(f"💾 Đang lưu kết quả vào: {output_file}")
    df_final.to_parquet(output_file, index=False)
    print("🎉 Hoàn thành!")
else:
    print("⚠️ Cảnh báo: File kết quả rỗng!")

df_final 

🧬 Đang tải FASTA: D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa
📊 Đang đọc file VEP CSV: D:\variant_data\test_seq_after_vep_final.parquet
⚙️ Bắt đầu xử lý 6095 biến thể...
  ...Đã xử lý 1000 / 6095...
  ...Đã xử lý 2000 / 6095...
  ...Đã xử lý 3000 / 6095...
  ...Đã xử lý 4000 / 6095...
  ...Đã xử lý 5000 / 6095...
  ...Đã xử lý 6000 / 6095...

🧹 Đang làm sạch dữ liệu cuối cùng...
Số missing value sau khi map: 

 - Tổng input: 6095
 - Thành công: 6095
 - Thất bại/Lỗi Verify: 0
 - Loại bỏ do nhiễm sắc thể rác: 0
✅ Tổng số hàng hợp lệ cuối cùng: 6095
💾 Đang lưu kết quả vào: D:\variant_data\test_seq_after_vep_final.parquet
🎉 Hoàn thành!


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,ref_seq_301,alt_seq_301,ref_seq_601,alt_seq_601,ref_seq_1001,alt_seq_1001,ref_seq_2001,alt_seq_2001,ref_seq_10001,alt_seq_10001
0,10_180088_C_T,424656.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.76C>T (p.Arg26Trp),10771.0,ZMYND11,HGNC:16966,Pathogenic/Likely pathogenic,1.0,"MONDO:MONDO:0014486,MedGen:C4015167,OMIM:61608...",...,TGATAATTCATTTTGTAATCTGTTTTTTTCCCTTATGTTTTTGTTT...,TGATAATTCATTTTGTAATCTGTTTTTTTCCCTTATGTTTTTGTTT...,AATGTCATTCTTCATGATGTAATGAAATGAATGATACTTTATATGA...,AATGTCATTCTTCATGATGTAATGAAATGAATGATACTTTATATGA...,AAACTAAAGTGCCAATTAGAATATTTAAATTCCATTTTGTATTGAT...,AAACTAAAGTGCCAATTAGAATATTTAAATTCCATTTTGTATTGAT...,ATGATAACATGTATGATCTAGAGAGATGTTTCAGTGCGTCCCCAAT...,ATGATAACATGTATGATCTAGAGAGATGTTTCAGTGCGTCCCCAAT...,TATACTGTGGACCTTGGGTCATAATGATATGTCAGTGTAGGTTTGT...,TATACTGTGGACCTTGGGTCATAATGATATGTCAGTGTAGGTTTGT...
1,10_237638_G_T,2038412.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.570G>T (p.Arg190Ser),10771.0,ZMYND11,HGNC:16966,Likely benign,0.0,"MedGen:C3661900|MeSH:D030342,MedGen:C0950123",...,AAAATAAAAAGGTAGAAAATATTTTGAGCTTTAGGAATTGATGTCT...,AAAATAAAAAGGTAGAAAATATTTTGAGCTTTAGGAATTGATGTCT...,TTGCTTTAAAGGTAGAAAATATTGGCCGGGCGCCGTGGCTCCCGCC...,TTGCTTTAAAGGTAGAAAATATTGGCCGGGCGCCGTGGCTCCCGCC...,AGCTACATGAGATTAGAAACATTTACATTTTTGAATTTCATTCATC...,AGCTACATGAGATTAGAAACATTTACATTTTTGAATTTCATTCATC...,TTTAAAGTTCTCACCTTGTTACATTAGATGCTGTTTTTTTTATGCA...,TTTAAAGTTCTCACCTTGTTACATTAGATGCTGTTTTTTTTATGCA...,TAGATCCCTCATTTATCACCAAAAAGGTGGGTGGGGCAACAAGTAG...,TAGATCCCTCATTTATCACCAAAAAGGTGGGTGGGGCAACAAGTAG...
2,10_240913_C_G,2077177.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.774C>G (p.Cys258Trp),10771.0,ZMYND11,HGNC:16966,Pathogenic,1.0,MedGen:C3661900,...,CTTAAAAGATTAAGAAACAAAATTCAACACTATTTATATACGTGAA...,CTTAAAAGATTAAGAAACAAAATTCAACACTATTTATATACGTGAA...,TCTGAAGTGCTAACCAGTGAGGTCCGGGCAGGGCCCAGTCAGACCA...,TCTGAAGTGCTAACCAGTGAGGTCCGGGCAGGGCCCAGTCAGACCA...,AGAATCGCTTGAACGTGGAAGGTGGAGGTTGCAGTGAGCCAAGATT...,AGAATCGCTTGAACGTGGAAGGTGGAGGTTGCAGTGAGCCAAGATT...,TATTTTGCTATCGTTATTTTAAATGGACATTTTGGTTTGAAATGGT...,TATTTTGCTATCGTTATTTTAAATGGACATTTTGGTTTGAAATGGT...,CTAAAAGTTCTCAGTTATGTGAGAAATATTTGTCCTTTCCTCTACT...,CTAAAAGTTCTCAGTTATGTGAGAAATATTTGTCCTTTCCTCTACT...
3,10_242031_A_C,NaN,None,None,NaN,None,None,None,NaN,None,...,AAAAATCTCGTATGTGTTTAGGAGTTATGAAAGAAATATTTTAGAA...,AAAAATCTCGTATGTGTTTAGGAGTTATGAAAGAAATATTTTAGAA...,GCTGGCTCCCCAGCTGCACTTGGCCAAGCGGGTGCTCTTGTTTGCC...,GCTGGCTCCCCAGCTGCACTTGGCCAAGCGGGTGCTCTTGTTTGCC...,AGACCTAAGGCTTTGCCTTTGTCTGTAGCTGTATAATTTCCCCAAA...,AGACCTAAGGCTTTGCCTTTGTCTGTAGCTGTATAATTTCCCCAAA...,TTGGTTAAAGATTATAATTTTCTTCCTAAATTTTTAAAATATCATA...,TTGGTTAAAGATTATAATTTTCTTCCTAAATTTTTAAAATATCATA...,TGGAGAGCACCCCGTCATGTCATATTTCTTTTTTGCAGTAAACAGT...,TGGAGAGCACCCCGTCATGTCATATTTCTTTTTTGCAGTAAACAGT...
4,10_246823_C_G,3002871.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.1008C>G (p.His336Gln),10771.0,ZMYND11,HGNC:16966,Benign,0.0,MedGen:C3661900,...,CTTTATGGCAGGCAGGGTCCACTGAAGCCCTCTTTTACCACCCTTC...,CTTTATGGCAGGCAGGGTCCACTGAAGCCCTCTTTTACCACCCTTC...,CAAATCTCATTTATTTTCCGCTTGGTAACAGTTTATTTATTCAAGC...,CAAATCTCATTTATTTTCCGCTTGGTAACAGTTTATTTATTCAAGC...,CAAGTGATTCTTTAGTGCTAACGGCAAATAGCTAGATGGGTTATAT...,CAAGTGATTCTTTAGTGCTAACGGCAAATAGCTAGATGGGTTATAT...,CTCAAAGGCAGAGGCAGTGTAGGAAGTGCCATCCTAATAATCAGAA...,CTCAAAGGCAGAGGCAGTGTAGGAAGTGCCATCCTAATAATCAGAA...,AGGAATAAGGTTCATGTTGGCACTCTGATGCAAATATAAAATCAGT...,AGGAATAAGGTTCATGTTGGCACTCTGATGCAAATATAAAATCAGT...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6090,20_64208061_G_A,728821.0,single nucleotide variant,NM_004535.3(MYT1):c.865G>A (p.Glu289Lys),4661.0,MYT1,HGNC:7622,Likely benign,0.0,MedGen:C3661900|,...,TCCCTGGAGGATGCAGCCAGTGAGGAGTCCAGCAAGCAGAAAGGCA...,TCCCTGGAGGATGCAGCCAGTGAGGAGTCCAGCAAGCAGAAAGGCA...,AAGCCAGGTCCTGGCATTGTGCACCTGCTTCAGGAGGCTGCAGAGG...,AAGCCAGGTCCTGGCATTGTGCACCTGCTTCAGGAGGCTGCAGAGG...,TCTGGCCCCCACGTTGATTTTGATTTTG

In [7]:
# --- 1. Cấu hình ---
fasta_path = r"D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa"
annotated_variant_file = r"D:\variant_data\clinvarhq_seq_after_vep_final.parquet"
output_file = r"D:\variant_data\clinvarhq_seq_after_vep_final.parquet"

# --- 3. Tải Dữ liệu ---
print(f"🧬 Đang tải FASTA: {fasta_path}")
try:
    genome = Fasta(fasta_path, as_raw=True, sequence_always_upper=True)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể tải file FASTA. Lỗi: {e}")
    sys.exit(1)

print(f"📊 Đang đọc file VEP CSV: {annotated_variant_file}")
try:
    df = pd.read_parquet(annotated_variant_file)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể đọc file Parquet. Lỗi: {e}")
    sys.exit(1)

# Khởi tạo các cột mới
df['canonical_center'] = None
df[f'ref_seq_{TARGET_LEN}'] = None
df[f'alt_seq_{TARGET_LEN}'] = None

count_success = 0
count_fail = 0

# --- 4. Vòng lặp Xử lý ---
total_rows = len(df)
print(f"⚙️ Bắt đầu xử lý {total_rows} biến thể...")
for index, row in df.iterrows():
    if (index + 1) % 1000 == 0:
        print(f"  ...Đã xử lý {index + 1} / {total_rows}...")

    try:
        # 1. Đọc dữ liệu (nếu đã qua filter)
        chrom = normalize_chrom(row['CHROM'])
        pos = int(row['POS'])
        ref_vcf = str(row['REF']).upper()
        alt_vcf = str(row['ALT']).upper()
        hgvsc_str = str(row['HGVSc'])
        consequence_str = str(row['Consequence'])
        # Lấy nhãn đầu tiên (nghiêm trọng nhất)
        first_consequence = consequence_str.split(',')[0]
        
        # 2. Tìm Tâm Chuẩn (Canonical Center)
        offset = parse_hgvsc_offset(hgvsc_str)
        canonical_center = 0

        if first_consequence == 'splice_donor_variant' and offset is not None:
            canonical_center = pos - (offset - 1)
        elif first_consequence == 'splice_acceptor_variant' and offset is not None:
            canonical_center = pos - (offset + 1)
        else:
            # Missense, UTR, hoặc Splicing bị lỗi offset -> Dùng POS làm tâm
            canonical_center = pos
        
        # 3. Tạo ref_seq
        ref_seq = get_ref_seq(genome, chrom, canonical_center, WINDOW)
        
        if not verify_ref_seq_center(genome, chrom, canonical_center, ref_seq, WINDOW):
            print(f"  CẢNH BÁO: Bỏ qua {chrom}:{pos} do lỗi xác minh căn giữa.")
            count_fail += 1
            continue
            
        # 4. Tạo alt_seq
        # Tính vị trí tương đối của biến thể so với tâm mới
        relative_pos = WINDOW + (pos - canonical_center)
        # Ghép chuỗi ALT
        alt_seq_dynamic = ref_seq[:relative_pos] + alt_vcf + ref_seq[relative_pos + len(ref_vcf):]
        indel_len_change = len(alt_vcf) - len(ref_vcf)
        alt_center_index = WINDOW
        # Nếu biến thể chính là tâm (Missense) -> relative_pos = WINDOW -> alt_center = WINDOW + change
        if relative_pos <= WINDOW:
            alt_center_index = WINDOW + indel_len_change
            
        final_alt_seq = normalize_centered_sequence(
            alt_seq_dynamic, 
            alt_center_index, 
            TARGET_LEN, 
            PAD_CHAR
        )

        # 5. Cập nhật vào DataFrame
        df.at[index, 'canonical_center'] = canonical_center
        df.at[index, f'ref_seq_{TARGET_LEN}'] = ref_seq
        df.at[index, f'alt_seq_{TARGET_LEN}'] = final_alt_seq
        
        count_success += 1
        
    except Exception as e:
        print(f"Lỗi xử lý hàng {index}: {e}. Dữ liệu hàng: {row.to_dict()}")

# --- 5. Hậu xử lý và Lưu file ---
print("\n🧹 Đang làm sạch dữ liệu cuối cùng...")

# Check data missing
print('Số missing value sau khi map: \n')
df.isnull().sum()

# 5.1. Loại bỏ những hàng không tạo được sequence (giá trị vẫn là None)
df_final = df.dropna(subset=[f'ref_seq_{TARGET_LEN}'])
# 5.2. Chuẩn hóa tên Chromosome output (hàm của bạn)
df_final['CHROM'] = df_final['CHROM'].apply(normalize_chromosome_output)
# 5.3. Loại bỏ các hàng CHROM bị None (do Un/random/alt)
before_clean = len(df_final)
df_final = df_final.dropna(subset=['CHROM'])
after_clean = len(df_final)

print(f" - Tổng input: {total_rows}")
print(f" - Thành công: {count_success}")
print(f" - Thất bại/Lỗi Verify: {count_fail}")
print(f" - Loại bỏ do nhiễm sắc thể rác: {before_clean - after_clean}")
print(f"✅ Tổng số hàng hợp lệ cuối cùng: {len(df_final)}")

# Lưu Kết quả
if not df_final.empty:
    print(f"💾 Đang lưu kết quả vào: {output_file}")
    df_final.to_parquet(output_file, index=False)
    print("🎉 Hoàn thành!")
else:
    print("⚠️ Cảnh báo: File kết quả rỗng!")

df_final 

🧬 Đang tải FASTA: D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa
📊 Đang đọc file VEP CSV: D:\variant_data\clinvarhq_seq_after_vep_final.parquet
⚙️ Bắt đầu xử lý 703 biến thể...

🧹 Đang làm sạch dữ liệu cuối cùng...
Số missing value sau khi map: 

 - Tổng input: 703
 - Thành công: 703
 - Thất bại/Lỗi Verify: 0
 - Loại bỏ do nhiễm sắc thể rác: 0
✅ Tổng số hàng hợp lệ cuối cùng: 703
💾 Đang lưu kết quả vào: D:\variant_data\clinvarhq_seq_after_vep_final.parquet
🎉 Hoàn thành!


,Variant_ID,CHROM,POS,REF,ALT,Label,ID,GeneInfo,CLNSIG,CLNREVSTAT,...,ref_seq_301,alt_seq_301,ref_seq_601,alt_seq_601,ref_seq_1001,alt_seq_1001,ref_seq_2001,alt_seq_2001,ref_seq_10001,alt_seq_10001
0,10_237638_G_T,chr10,237638,G,T,0,1980541,ZMYND11,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,AAAATAAAAAGGTAGAAAATATTTTGAGCTTTAGGAATTGATGTCT...,AAAATAAAAAGGTAGAAAATATTTTGAGCTTTAGGAATTGATGTCT...,TTGCTTTAAAGGTAGAAAATATTGGCCGGGCGCCGTGGCTCCCGCC...,TTGCTTTAAAGGTAGAAAATATTGGCCGGGCGCCGTGGCTCCCGCC...,AGCTACATGAGATTAGAAACATTTACATTTTTGAATTTCATTCATC...,AGCTACATGAGATTAGAAACATTTACATTTTTGAATTTCATTCATC...,TTTAAAGTTCTCACCTTGTTACATTAGATGCTGTTTTTTTTATGCA...,TTTAAAGTTCTCACCTTGTTACATTAGATGCTGTTTTTTTTATGCA...,TAGATCCCTCATTTATCACCAAAAAGGTGGGTGGGGCAACAAGTAG...,TAGATCCCTCATTTATCACCAAAAAGGTGGGTGGGGCAACAAGTAG...
1,10_357852_G_C,chr10,357852,G,C,0,2758294,DIP2C,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,AGACACTGTCGCGGACTGTCAGGGAAGGTAGGGGACAGTCGGGTAC...,AGACACTGTCGCGGACTGTCAGGGAAGGTAGGGGACAGTCGGGTAC...,TAAACACATTTTTCTTATGAAATTATATGGTCTCTTGCAGGAGAGG...,TAAACACATTTTTCTTATGAAATTATATGGTCTCTTGCAGGAGAGG...,AGAAGCCTTGCTGCAAGCTGCTGTGGGCAGGGTGGGCTTTGAATCC...,AGAAGCCTTGCTGCAAGCTGCTGTGGGCAGGGTGGGCTTTGAATCC...,TGCACAGAAATCCCTTCTCCTGGGGCTTCCAGAAGATGCAAAGCGG...,TGCACAGAAATCCCTTCTCCTGGGGCTTCCAGAAGATGCAAAGCGG...,TCCTCAGCCTTGCTCCCCAGGAATCCTAATGCAGCCTCTACCCCAG...,TCCTCAGCCTTGCTCCCCAGGAATCCTAATGCAGCCTCTACCCCAG...
2,10_1086292_C_T,chr10,1086292,C,T,0,2279214,WDR37,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,CATACAGAGTAATAGAATCAAAGGTGGAAGAGATATCAGTGGTCGT...,CATACAGAGTAATAGAATCAAAGGTGGAAGAGATATCAGTGGTCGT...,CGACCTCTGTCTGTCATTTTCCAGCATGATGCAGCCAGCCTGATAG...,CGACCTCTGTCTGTCATTTTCCAGCATGATGCAGCCAGCCTGATAG...,GGCTAATTGGTGCTTATTTCTCAGATTTTCCCATTAAATCTACTTA...,GGCTAATTGGTGCTTATTTCTCAGATTTTCCCATTAAATCTACTTA...,CAAGTGTGTTTCTGATTTGTGGATGTATTTGACTTTTTTCCCACAA...,CAAGTGTGTTTCTGATTTGTGGATGTATTTGACTTTTTTCCCACAA...,GATGGATTACTATCAAGTGATATTTTGAGGTAATTTTGTATGCATC...,GATGGATTACTATCAAGTGATATTTTGAGGTAATTTTGTATGCATC...
3,10_5102114_C_T,chr10,5102114,C,T,0,716665,AKR1C3,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,TTATTACTTGACAATAATATCCTCAGCTCAAATATAATGCTATACT...,TTATTACTTGACAATAATATCCTCAGCTCAAATATAATGCTATACT...,GTATCCCAGATATGGAACTTGTTACATCTCCTTCTAGTTGTCAAAG...,GTATCCCAGATATGGAACTTGTTACATCTCCTTCTAGTTGTCAAAG...,TTAATTTTATTAATTTTTGTTAGTGCTCATTCTTACCCAATGGGTA...,TTAATTTTATTAATTTTTGTTAGTGCTCATTCTTACCCAATGGGTA...,ATGGAAATGTCTTTTATAGTGCTCCCTTTTAAAACAGAATGGAAAC...,ATGGAAATGTCTTTTATAGTGCTCCCTTTTAAAACAGAATGGAAAC...,AAGAATTAGACTGTTAAAATGAGTTGTAAACTCTACTGAAGATAAT...,AAGAATTAGACTGTTAAAATGAGTTGTAAACTCTACTGAAGATAAT...
4,10_8064041_G_A,chr10,8064041,G,A,1,3384342,GATA3,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,AAAAAAGTTCTCCATTTTACGTTTCTCCCCCAGTTCCAAATGCTGT...,AAAAAAGTTCTCCATTTTACGTTTCTCCCCCAGTTCCAAATGCTGT...,TCTGTTGCAACGATGCATCTGCCCCTTCTGCGGGCGCCTCCGTGTG...,TCTGTTGCAACGATGCATCTGCCCCTTCTGCGGGCGCCTCCGTGTG...,TCTCTATAGACAGACCCATGATGATAATGGGTCACCTTTGTCTTCC...,TCTCTATAGACAGACCCATGATGATAATGGGTCACCTTTGTCTTCC...,CTGCTCAGACTTCCTACCGGCCTGGCTGGTTTTTCGTTTATTTTGT...,CTGCTCAGACTTCCTACCGGCCTGGCTGGTTTTTCGTTTATTTTGT...,GAAGTGGGATCTGATTTAAAACCCCCCAATGAGCTGGGATAGGAAA...,GAAGTGGGATCTGATTTAAAACCCCCCAATGAGCTGGGATAGGAAA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
698,20_63488381_C_A,chr20,63488381,C,A,1,383531,EEF1A2,Likely_pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,GGGGGCCCGGGCCCGGGGTTCGGAGCGCGGCACCGCCGGGGAGGGT...,GGGGGCCCGGGCCCGGGGTTCGGAGCGCGGCACCGCCGGGGAGGGT...,GCGGGGGCGCCTTTCCTCTTGAAGAACTTCCACTGGACCTTGATGG...,GCGGGGGCGCCTTTCCTCTTGAAGAACTTCCACTGGACCTTGATGG...,ACCACTCTTGCAGGCCCACCGCCCACCCTGGGGTCCCCGGGCAGGG...,ACCACTCTTGCAGGCCCACCGCCCACCCTGGGGTCCCCGGGCAGGG...,TCTCCCCACCCCTCCACTCTTCCCAGCCAAGGTCGGGGCAGGTTCT...,TCTCCCCACCCCTCCACTCTTCCCAGCCAAGGTCGGGGCAGGTTCT...,CCTTCCCGCCTGGGTTACAGGAGGGCACCGGGAGCCCGGGAGAGAC...,CCTTCCCGCCTGGGTTACAGGAGGGCACCGGGAGC

In [8]:
# --- 1. Cấu hình ---
fasta_path = r"D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa"
annotated_variant_file = r"D:\variant_data\uniprot_seq_after_vep_final.parquet"
output_file = r"D:\variant_data\uniprot_seq_after_vep_final.parquet"
    
# --- 3. Tải Dữ liệu ---
print(f"🧬 Đang tải FASTA: {fasta_path}")
try:
    genome = Fasta(fasta_path, as_raw=True, sequence_always_upper=True)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể tải file FASTA. Lỗi: {e}")
    sys.exit(1)

print(f"📊 Đang đọc file VEP CSV: {annotated_variant_file}")
try:
    df = pd.read_parquet(annotated_variant_file)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể đọc file Parquet. Lỗi: {e}")
    sys.exit(1)

# Khởi tạo các cột mới
df['canonical_center'] = None
df[f'ref_seq_{TARGET_LEN}'] = None
df[f'alt_seq_{TARGET_LEN}'] = None

count_success = 0
count_fail = 0

# --- 4. Vòng lặp Xử lý ---
total_rows = len(df)
print(f"⚙️ Bắt đầu xử lý {total_rows} biến thể...")
for index, row in df.iterrows():
    if (index + 1) % 1000 == 0:
        print(f"  ...Đã xử lý {index + 1} / {total_rows}...")

    try:
        # 1. Đọc dữ liệu (nếu đã qua filter)
        chrom = normalize_chrom(row['CHROM'])
        pos = int(row['POS'])
        ref_vcf = str(row['REF']).upper()
        alt_vcf = str(row['ALT']).upper()
        hgvsc_str = str(row['HGVSc'])
        consequence_str = str(row['Consequence'])
        # Lấy nhãn đầu tiên (nghiêm trọng nhất)
        first_consequence = consequence_str.split(',')[0]
        
        # 2. Tìm Tâm Chuẩn (Canonical Center)
        offset = parse_hgvsc_offset(hgvsc_str)
        canonical_center = 0

        if first_consequence == 'splice_donor_variant' and offset is not None:
            canonical_center = pos - (offset - 1)
        elif first_consequence == 'splice_acceptor_variant' and offset is not None:
            canonical_center = pos - (offset + 1)
        else:
            # Missense, UTR, hoặc Splicing bị lỗi offset -> Dùng POS làm tâm
            canonical_center = pos
        
        # 3. Tạo ref_seq
        ref_seq = get_ref_seq(genome, chrom, canonical_center, WINDOW)
        
        if not verify_ref_seq_center(genome, chrom, canonical_center, ref_seq, WINDOW):
            print(f"  CẢNH BÁO: Bỏ qua {chrom}:{pos} do lỗi xác minh căn giữa.")
            count_fail += 1
            continue
            
        # 4. Tạo alt_seq
        # Tính vị trí tương đối của biến thể so với tâm mới
        relative_pos = WINDOW + (pos - canonical_center)
        # Ghép chuỗi ALT
        alt_seq_dynamic = ref_seq[:relative_pos] + alt_vcf + ref_seq[relative_pos + len(ref_vcf):]
        indel_len_change = len(alt_vcf) - len(ref_vcf)
        alt_center_index = WINDOW
        # Nếu biến thể chính là tâm (Missense) -> relative_pos = WINDOW -> alt_center = WINDOW + change
        if relative_pos <= WINDOW:
            alt_center_index = WINDOW + indel_len_change
            
        final_alt_seq = normalize_centered_sequence(
            alt_seq_dynamic, 
            alt_center_index, 
            TARGET_LEN, 
            PAD_CHAR
        )

        # 5. Cập nhật vào DataFrame
        df.at[index, 'canonical_center'] = canonical_center
        df.at[index, f'ref_seq_{TARGET_LEN}'] = ref_seq
        df.at[index, f'alt_seq_{TARGET_LEN}'] = final_alt_seq
        
        count_success += 1
        
    except Exception as e:
        print(f"Lỗi xử lý hàng {index}: {e}. Dữ liệu hàng: {row.to_dict()}")

# --- 5. Hậu xử lý và Lưu file ---
print("\n🧹 Đang làm sạch dữ liệu cuối cùng...")

# Check data missing
print('Số missing value sau khi map: \n')
df.isnull().sum()

# 5.1. Loại bỏ những hàng không tạo được sequence (giá trị vẫn là None)
df_final = df.dropna(subset=[f'ref_seq_{TARGET_LEN}'])
# 5.2. Chuẩn hóa tên Chromosome output (hàm của bạn)
df_final['CHROM'] = df_final['CHROM'].apply(normalize_chromosome_output)
# 5.3. Loại bỏ các hàng CHROM bị None (do Un/random/alt)
before_clean = len(df_final)
df_final = df_final.dropna(subset=['CHROM'])
after_clean = len(df_final)

print(f" - Tổng input: {total_rows}")
print(f" - Thành công: {count_success}")
print(f" - Thất bại/Lỗi Verify: {count_fail}")
print(f" - Loại bỏ do nhiễm sắc thể rác: {before_clean - after_clean}")
print(f"✅ Tổng số hàng hợp lệ cuối cùng: {len(df_final)}")

# Lưu Kết quả
if not df_final.empty:
    print(f"💾 Đang lưu kết quả vào: {output_file}")
    df_final.to_parquet(output_file, index=False)
    print("🎉 Hoàn thành!")
else:
    print("⚠️ Cảnh báo: File kết quả rỗng!")

df_final 

🧬 Đang tải FASTA: D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa
📊 Đang đọc file VEP CSV: D:\variant_data\uniprot_seq_after_vep_final.parquet
⚙️ Bắt đầu xử lý 1333 biến thể...
  ...Đã xử lý 1000 / 1333...

🧹 Đang làm sạch dữ liệu cuối cùng...
Số missing value sau khi map: 

 - Tổng input: 1333
 - Thành công: 1333
 - Thất bại/Lỗi Verify: 0
 - Loại bỏ do nhiễm sắc thể rác: 0
✅ Tổng số hàng hợp lệ cuối cùng: 1333
💾 Đang lưu kết quả vào: D:\variant_data\uniprot_seq_after_vep_final.parquet
🎉 Hoàn thành!


,Variant_ID,CHROM,POS,REF,ALT,Label,dbSNP,gene,protein_AC,aa_change,...,ref_seq_301,alt_seq_301,ref_seq_601,alt_seq_601,ref_seq_1001,alt_seq_1001,ref_seq_2001,alt_seq_2001,ref_seq_10001,alt_seq_10001
0,10_1072186_G_A,chr10,1072186,G,A,0,rs17856557,WDR37,Q9Y2I8,p.Ala11Thr,...,GTCATCATTATCTTCTTCACTGTTTTTCTTTTGTTTCAACTCGCTG...,GTCATCATTATCTTCTTCACTGTTTTTCTTTTGTTTCAACTCGCTG...,TGCCATTTGATACTTAAGATGTTAATGAAATTTGATAAAGAAATTA...,TGCCATTTGATACTTAAGATGTTAATGAAATTTGATAAAGAAATTA...,GCTTTTATTTTTAAAAATGTGGACACTGGAAAATTTACATCCATGA...,GCTTTTATTTTTAAAAATGTGGACACTGGAAAATTTACATCCATGA...,AGATGATGGAGGCATGTTGGCGTCTGGGAGGTCGTGGGCGTGTCCT...,AGATGATGGAGGCATGTTGGCGTCTGGGAGGTCGTGGGCGTGTCCT...,TCTAGGATGGATCATCTTCAACAAATGAGGCTGGGATAATTGGATA...,TCTAGGATGGATCATCTTCAACAAATGAGGCTGGGATAATTGGATA...
1,10_1096193_A_G,chr10,1096193,A,G,0,rs2306407,WDR37,Q9Y2I8,p.Ile225Val,...,ATTCCTCTCACTAAGTGGTCACTAAGCTGGAAAGGTCTGACTGCTG...,ATTCCTCTCACTAAGTGGTCACTAAGCTGGAAAGGTCTGACTGCTG...,ACATTTTCTCTGAACTTCCAGGACTTTCTATGCTAAAATGAAAGGT...,ACATTTTCTCTGAACTTCCAGGACTTTCTATGCTAAAATGAAAGGT...,TGCTGAAAACAGAGTGGGACGGGAGCCGCACACATGAGCAGTGCTG...,TGCTGAAAACAGAGTGGGACGGGAGCCGCACACATGAGCAGTGCTG...,GTGAGGTGACGAGCATGGAGAGAGGAGAGTTAGACTTGGAAACGTG...,GTGAGGTGACGAGCATGGAGAGAGGAGAGTTAGACTTGGAAACGTG...,CACACCTCACAGTGGCCCAGCTTCGTTTGTGATTAGCTGAGGGTTG...,CACACCTCACAGTGGCCCAGCTTCGTTTGTGATTAGCTGAGGGTTG...
2,10_1379131_C_A,chr10,1379131,C,A,0,rs3793733,ADARB2,Q9NS39,p.Ala44Thr,...,ACCCCAGGTATCTCAGGTTTTGGGGACTGCAGGGGACCCCTGCACA...,ACCCCAGGTATCTCAGGTTTTGGGGACTGCAGGGGACCCCTGCACA...,GGGTGAGGACAGGACCTCAAATAAGAAAAGGATTCCAGCTGAGGAC...,GGGTGAGGACAGGACCTCAAATAAGAAAAGGATTCCAGCTGAGGAC...,ACAGGTGAGGATGAGACTGCAGGTGTGGGTAAGACTCCAGGTGCGG...,ACAGGTGAGGATGAGACTGCAGGTGTGGGTAAGACTCCAGGTGCGG...,ATGCCTGACAATTTGCTGTGAACCAGAGGCCAGCTCCTGCAGCCAC...,ATGCCTGACAATTTGCTGTGAACCAGAGGCCAGCTCCTGCAGCCAC...,AACTGTGGCTTCTCTGAGGTCTTTGTGGGCGGCACCAGGTTTTCAG...,AACTGTGGCTTCTCTGAGGTCTTTGTGGGCGGCACCAGGTTTTCAG...
3,10_3151324_G_A,chr10,3151324,G,A,0,rs12248937,PITRM1,Q5JRX3,p.Ala554Asp,...,GCTTCTGTGAGGAGTGACATTTAATTCCCCCAAGGAACCCGAGACC...,GCTTCTGTGAGGAGTGACATTTAATTCCCCCAAGGAACCCGAGACC...,GAACACACACACACACGGGCTCTTACTGAGTGAAGGAGTCTATCAC...,GAACACACACACACACGGGCTCTTACTGAGTGAAGGAGTCTATCAC...,CCTTTCAACTCAGCACTCAGCTTTCTGGGGGCTCTCGTCCCTCTGC...,CCTTTCAACTCAGCACTCAGCTTTCTGGGGGCTCTCGTCCCTCTGC...,TTAATGAGATTCACGGACAAAAACTGCCAGGGTTAACAGATCAGAG...,TTAATGAGATTCACGGACAAAAACTGCCAGGGTTAACAGATCAGAG...,AACTATATCCCAGGCATGTATGCTGCCTTGCAGATATTCTTTCTTT...,AACTATATCCCAGGCATGTATGCTGCCTTGCAGATATTCTTTCTTT...
4,10_3165320_C_T,chr10,3165320,C,T,1,rs1249144069,PITRM1,Q5JRX3,p.Arg183Gln,...,GTCTATATCATGATATTGTACATGGGAAAGGTAAGCTGAACACAAC...,GTCTATATCATGATATTGTACATGGGAAAGGTAAGCTGAACACAAC...,ATACCAGGGGTCCCCAGACATGGTCTGAGACCTGCTGGGAAGAACA...,ATACCAGGGGTCCCCAGACATGGTCTGAGACCTGCTGGGAAGAACA...,TTGCAAATCTCTGTAGCGTCTGGCTCAGAGGTGAAGCTGGACTCCC...,TTGCAAATCTCTGTAGCGTCTGGCTCAGAGGTGAAGCTGGACTCCC...,GGAATCAGAGAGCAGGCCACCCATGAGCCTCAGGGACACCAGGACA...,GGAATCAGAGAGCAGGCCACCCATGAGCCTCAGGGACACCAGGACA...,ACGTGAAGAACCTAAAATTTTAAGGGAATATAATTAGTCTGTTTTA...,ACGTGAAGAACCTAAAATTTTAAGGGAATATAATTAGTCTGTTTTA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1328,20_63495972_C_T,chr20,63495972,C,T,1,rs587777162,EEF1A2,Q05639,p.Gly70Ser,...,TCTCCCCCAGCCCCGCCTGCTGTGCCCTGCTCACCTGGGATGTACC...,TCTCCCCCAGCCCCGCCTGCTGTGCCCTGCTCACCTGGGATGTACC...,AGCACTGGATTCATCCTTAGGGGGGCTCTGAGCCAGACTGGGTGAG...,AGCACTGGATTCATCCTTAGGGGGGCTCTGAGCCAGACTGGGTGAG...,TCCGTGCGTCTCTGTGAGTGCCCTGAAATGTCATACAGTCTACACC...,TCCGTGCGTCTCTGTGAGTGCCCTGAAATGTCATACAGTCTACACC...,CGCCCACGATGAGCTGCTTCACACCCAGCGTGTAGGCCAGCAGGGC...,CGCCCACGATGAGCTGCTTCACACCCAGCGTGTAGGCCAGCAGGGC...,CAGCAGAGCTCCCAGGAACCCACGGAAAAGCCACAGGGACAGAGAA...,CAGCAGAGCTCCCAGGAACCCACGGAAAAGCCACAGGGACAGAGAA...
1329,20_63547241_C_A,chr20,63547241,C,A,0,rs55863722,SRMS,Q9H3Y6,p.Gly75Arg,...,GAGGCCGAGGTACTCACGGTTGGTCTGAGAGCGTCTCAGGAGAAGC...,GAGGCCGAGGTACTCACGGTTGGTCTGAGAGCGTCTCAGGAGAAGC...,ACCTGCCGGCCCGGGGCCCATCAGCTGTCCCTGAATGAATGCGTGG...,ACCTGCCGGCCCG

In [9]:
# --- 1. Cấu hình ---
fasta_path = r"D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa"
annotated_variant_file = r"D:\variant_data\proteingym_after_vep_final.parquet"
output_file = r"D:\variant_data\proteingym_seq_seq_after_vep_final.parquet"

# --- 3. Tải Dữ liệu ---
print(f"🧬 Đang tải FASTA: {fasta_path}")
try:
    genome = Fasta(fasta_path, as_raw=True, sequence_always_upper=True)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể tải file FASTA. Lỗi: {e}")
    sys.exit(1)

print(f"📊 Đang đọc file VEP CSV: {annotated_variant_file}")
try:
    df = pd.read_parquet(annotated_variant_file)
except Exception as e:
    print(f"Lỗi nghiêm trọng: Không thể đọc file Parquet. Lỗi: {e}")
    sys.exit(1)

# Khởi tạo các cột mới
df['canonical_center'] = None
df[f'ref_seq_{TARGET_LEN}'] = None
df[f'alt_seq_{TARGET_LEN}'] = None

count_success = 0
count_fail = 0

# --- 4. Vòng lặp Xử lý ---
total_rows = len(df)
print(f"⚙️ Bắt đầu xử lý {total_rows} biến thể...")
for index, row in df.iterrows():
    if (index + 1) % 1000 == 0:
        print(f"  ...Đã xử lý {index + 1} / {total_rows}...")

    try:
        # 1. Đọc dữ liệu (nếu đã qua filter)
        chrom = normalize_chrom(row['CHROM'])
        pos = int(row['POS'])
        ref_vcf = str(row['REF']).upper()
        alt_vcf = str(row['ALT']).upper()
        hgvsc_str = str(row['HGVSc'])
        consequence_str = str(row['Consequence'])
        # Lấy nhãn đầu tiên (nghiêm trọng nhất)
        first_consequence = consequence_str.split(',')[0]
        
        # 2. Tìm Tâm Chuẩn (Canonical Center)
        offset = parse_hgvsc_offset(hgvsc_str)
        canonical_center = 0

        if first_consequence == 'splice_donor_variant' and offset is not None:
            canonical_center = pos - (offset - 1)
        elif first_consequence == 'splice_acceptor_variant' and offset is not None:
            canonical_center = pos - (offset + 1)
        else:
            # Missense, UTR, hoặc Splicing bị lỗi offset -> Dùng POS làm tâm
            canonical_center = pos
        
        # 3. Tạo ref_seq
        ref_seq = get_ref_seq(genome, chrom, canonical_center, WINDOW)
        
        if not verify_ref_seq_center(genome, chrom, canonical_center, ref_seq, WINDOW):
            print(f"  CẢNH BÁO: Bỏ qua {chrom}:{pos} do lỗi xác minh căn giữa.")
            count_fail += 1
            continue
            
        # 4. Tạo alt_seq
        # Tính vị trí tương đối của biến thể so với tâm mới
        relative_pos = WINDOW + (pos - canonical_center)
        # Ghép chuỗi ALT
        alt_seq_dynamic = ref_seq[:relative_pos] + alt_vcf + ref_seq[relative_pos + len(ref_vcf):]
        indel_len_change = len(alt_vcf) - len(ref_vcf)
        alt_center_index = WINDOW
        # Nếu biến thể chính là tâm (Missense) -> relative_pos = WINDOW -> alt_center = WINDOW + change
        if relative_pos <= WINDOW:
            alt_center_index = WINDOW + indel_len_change
            
        final_alt_seq = normalize_centered_sequence(
            alt_seq_dynamic, 
            alt_center_index, 
            TARGET_LEN, 
            PAD_CHAR
        )

        # 5. Cập nhật vào DataFrame
        df.at[index, 'canonical_center'] = canonical_center
        df.at[index, f'ref_seq_{TARGET_LEN}'] = ref_seq
        df.at[index, f'alt_seq_{TARGET_LEN}'] = final_alt_seq
        
        count_success += 1
        
    except Exception as e:
        print(f"Lỗi xử lý hàng {index}: {e}. Dữ liệu hàng: {row.to_dict()}")

# --- 5. Hậu xử lý và Lưu file ---
print("\n🧹 Đang làm sạch dữ liệu cuối cùng...")

# Check data missing
print('Số missing value sau khi map: \n')
df.isnull().sum()

# 5.1. Loại bỏ những hàng không tạo được sequence (giá trị vẫn là None)
df_final = df.dropna(subset=[f'ref_seq_{TARGET_LEN}'])
# 5.2. Chuẩn hóa tên Chromosome output (hàm của bạn)
df_final['CHROM'] = df_final['CHROM'].apply(normalize_chromosome_output)
# 5.3. Loại bỏ các hàng CHROM bị None (do Un/random/alt)
before_clean = len(df_final)
df_final = df_final.dropna(subset=['CHROM'])
after_clean = len(df_final)

print(f" - Tổng input: {total_rows}")
print(f" - Thành công: {count_success}")
print(f" - Thất bại/Lỗi Verify: {count_fail}")
print(f" - Loại bỏ do nhiễm sắc thể rác: {before_clean - after_clean}")
print(f"✅ Tổng số hàng hợp lệ cuối cùng: {len(df_final)}")

# Lưu Kết quả
if not df_final.empty:
    print(f"💾 Đang lưu kết quả vào: {output_file}")
    df_final.to_parquet(output_file, index=False)
    print("🎉 Hoàn thành!")
else:
    print("⚠️ Cảnh báo: File kết quả rỗng!")

df_final 

🧬 Đang tải FASTA: D:\vep_resources\Homo_sapiens.GRCh38.dna.primary_assembly.fa
📊 Đang đọc file VEP CSV: D:\variant_data\proteingym_after_vep_final.parquet
⚙️ Bắt đầu xử lý 1472 biến thể...
  ...Đã xử lý 1000 / 1472...

🧹 Đang làm sạch dữ liệu cuối cùng...
Số missing value sau khi map: 

 - Tổng input: 1472
 - Thành công: 1472
 - Thất bại/Lỗi Verify: 0
 - Loại bỏ do nhiễm sắc thể rác: 0
✅ Tổng số hàng hợp lệ cuối cùng: 1472
💾 Đang lưu kết quả vào: D:\variant_data\proteingym_seq_seq_after_vep_final.parquet
🎉 Hoàn thành!


,Variant_ID,CHROM,POS,REF,ALT,Label,protein,protein_sequence,mutant,mutated_sequence,...,REVEL_pred,CADD_pred,DANN_pred,gMVP_pred,MPC_pred,VARITY_R_pred,VARITY_ER_pred,canonical_center,ref_seq_10001,alt_seq_10001
0,10_180088_C_T,chr10,180088,C,T,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,R26W,MARLTKRRQADTKAIQHLWAAIEIIWNQKQIANIDRITKYMSRVHG...,...,T,D,D,D,D,T,T,180088,TATACTGTGGACCTTGGGTCATAATGATATGTCAGTGTAGGTTTGT...,TATACTGTGGACCTTGGGTCATAATGATATGTCAGTGTAGGTTTGT...
1,10_240913_C_G,chr10,240913,C,G,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,C258W,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,T,D,D,D,D,D,T,240913,CTAAAAGTTCTCAGTTATGTGAGAAATATTTGTCCTTTCCTCTACT...,CTAAAAGTTCTCAGTTATGTGAGAAATATTTGTCCTTTCCTCTACT...
2,10_242031_A_C,chr10,242031,A,C,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,H281P,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,D,D,D,D,D,D,D,242031,TGGAGAGCACCCCGTCATGTCATATTTCTTTTTTGCAGTAAACAGT...,TGGAGAGCACCCCGTCATGTCATATTTCTTTTTTGCAGTAAACAGT...
3,10_248502_G_A,chr10,248502,G,A,0,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,S465N,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,T,T,D,T,T,T,T,248502,GAGTGATATTAATAATGAGTTAACTTGTTAGATGTCATTCAGTATA...,GAGTGATATTAATAATGAGTTAACTTGTTAGATGTCATTCAGTATA...
4,10_248559_C_T,chr10,248559,C,T,0,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,S484L,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,T,D,D,T,T,D,T,248559,ATATTTTTAAAGAATATTTGGGTTAGGCCAGGCACGGTGGCTCACG...,ATATTTTTAAAGAATATTTGGGTTAGGCCAGGCACGGTGGCTCACG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1467,20_64207869_G_A,chr20,64207869,G,A,0,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,V225I,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,T,T,T,T,T,T,T,64207869,AAAGAGGCTCGGGAGTCCTGGAGCCCTGGCCGTCTTCCTCTCCCTA...,AAAGAGGCTCGGGAGTCCTGGAGCCCTGGCCGTCTTCCTCTCCCTA...
1468,20_64207986_G_C,chr20,64207986,G,C,1,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,E264Q,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,T,T,D,T,T,T,T,64207986,GCTGGCATCTGCTGGCACGACACCCTTGGAGGCCCTGGCTGGGTCT...,GCTGGCATCTGCTGGCACGACACCCTTGGAGGCCCTGGCTGGGTCT...
1469,20_64208061_G_A,chr20,64208061,G,A,0,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,E289K,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,T,D,T,T,T,T,T,64208061,AGGCATCTAAGGGGTTTCCAGCTTCCTGTCCCCGAGCGGCCTTGTC...,AGGCATCTAAGGGGTTTCCAGCTTCCTGTCCCCGAGCGGCCTTGTC...
1470,20_64208272_G_A,chr20,64208272,G,A,0,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,R359Q,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,T,T,T,T,T,T,T,64208272,TGATTCACTCAGTGATGTCATCGGCTAATTGGGACCTTAGCTTCAA...,TGATTCACTCAGTGATGTCATCGGCTAATTGGGACCTTAGCTTCAA...
